# GAIA TELESCOPE — Intelligent Query Pipeline
Guillem Masdemont, Plabon Shaha, Pietro Sestito

## 1 · Environment setup & imports

In [1]:
#pip install import-ipynb

In [2]:
# ── SSL fix for ARNES cluster ─────────────────────────────────────────────────
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import os
os.environ.setdefault('CURL_CA_BUNDLE', '')
os.environ.setdefault('REQUESTS_CA_BUNDLE', '')

# ── Standard imports ──────────────────────────────────────────────────────────
import json
import re
import time
import numpy as np
import pandas as pd
from IPython.display import display

print('Imports OK')

Imports OK


## 2 · HuggingFace cache & SSL

In [3]:
HF_CACHE = os.path.abspath('../hf_cache')
os.environ['HF_HOME']            = HF_CACHE
os.environ['HF_HUB_CACHE']       = os.path.join(HF_CACHE, 'hub')
os.environ['HF_HUB_DISABLE_XET'] = '1'

ssl._create_default_https_context = ssl._create_unverified_context
urllib3.disable_warnings()
os.environ['CURL_CA_BUNDLE']     = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''

print(f'HF cache: {HF_CACHE}')

HF cache: /d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache


## 3 · Load vLLM model

In [4]:
from vllm import LLM, SamplingParams

MODEL_ID             = 'Qwen/Qwen2.5-7B-Instruct'
TENSOR_PARALLEL_SIZE = int(os.environ.get('TENSOR_PARALLEL_SIZE', '1'))
DTYPE                = 'float16'

if 'llm' not in globals():
    print(f'Loading model from {HF_CACHE} …')
    llm = LLM(
        model=MODEL_ID,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        dtype=DTYPE,
        gpu_memory_utilization=0.90,
        trust_remote_code=True,
    )
    print('Model loaded.')
else:
    print('Model already loaded — reusing.')

SAMPLING_PARAMS       = SamplingParams(temperature=0.0, max_tokens=512)
JUDGE_SAMPLING_PARAMS = SamplingParams(temperature=0.0, max_tokens=512)

/usr/local/lib/python3.11/dist-packages/transformers/utils/hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading model from /d/hpc/projects/onj_fri/no-language-processors-v2/hf_cache …


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

WARNING 05-11 17:53:44 config.py:2276] Casting torch.bfloat16 to torch.float16.
INFO 05-11 17:53:56 config.py:510] This model supports multiple tasks: {'score', 'reward', 'embed', 'classify', 'generate'}. Defaulting to 'generate'.
INFO 05-11 17:53:56 llm_engine.py:234] Initializing an LLM engine (v0.6.6.post1) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forwa

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

INFO 05-11 17:54:00 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 05-11 17:54:00 selector.py:129] Using XFormers backend.
INFO 05-11 17:54:01 model_runner.py:1094] Starting to load model Qwen/Qwen2.5-7B-Instruct...
INFO 05-11 17:54:02 weight_utils.py:251] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 05-11 17:54:41 model_runner.py:1099] Loading model weights took 14.2487 GB
INFO 05-11 17:54:47 worker.py:241] Memory profiling takes 6.24 seconds
INFO 05-11 17:54:47 worker.py:241] the current vLLM instance can use total_gpu_memory (31.73GiB) x gpu_memory_utilization (0.90) = 28.56GiB
INFO 05-11 17:54:47 worker.py:241] model weights take 14.25GiB; non_torch_memory takes 0.13GiB; PyTorch activation peak memory takes 4.35GiB; the rest of the memory reserved for KV Cache is 9.82GiB.
INFO 05-11 17:54:48 gpu_executor.py:76] # GPU blocks: 11497, # CPU blocks: 4681
INFO 05-11 17:54:48 gpu_executor.py:80] Maximum concurrency for 32768 tokens per request: 5.61x
INFO 05-11 17:54:53 model_runner.py:1415] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utiliz

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:29<00:00,  1.20it/s]

INFO 05-11 17:55:22 model_runner.py:1535] Graph capturing finished in 29 secs, took 0.20 GiB
INFO 05-11 17:55:22 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 41.09 seconds
Model loaded.


## 4 · LLM query parser and ADQL builder

In [5]:
# ── Valid Gaia DR3 columns the LLM may choose ─────────────────────────────────
VALID_COLUMNS = {
    'ra', 'dec', 'source_id', 'parallax', 'parallax_error',
    'pmra', 'pmdec', 'radial_velocity',
    'phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', 'bp_rp',
    'teff_gspphot', 'logg_gspphot', 'mh_gspÇphot',
    'phot_variable_flag', 'ruwe',
}

# ── Gaia DR3 internal tables available for crossmatch ────────────────────────
VALID_JOIN_TABLES = {
    'astrophysical_parameters',
    'vari_classifier_result',
    'vari_rrlyrae',
    'vari_cepheid',
    'vari_eclipsing_binary',
    'vari_rotation_modulation',
}

# ── Valid intents ─────────────────────────────────────────────────────────────
VALID_INTENTS = {
    'cone_search',          # Stars in a circular sky region
    'color_histogram',      # Colour distribution in a region
    'hr_diagram',           # Hertzsprung-Russell diagram (forces parallax+G+bp_rp)
    'stellar_population',   # Filter by star type (teff, logg, metallicity) in a region
    'variability_search',   # Variable stars in a region (phot_variable_flag, no JOIN)
    'internal_crossmatch',  # Join gaia_source with another DR3 table in a region
    'nearest_neighbours',   # Closest stars to the Sun (sky-wide, ORDER BY parallax)
    'velocity_computation', # 3D kinematics: pm + radial_velocity (cone optional)
}

# ── Column aliases the LLM might use ─────────────────────────────────────────
_COLUMN_ALIASES = {
    'magnitude':         'phot_g_mean_mag',
    'g_magnitude':       'phot_g_mean_mag',
    'g_mag':             'phot_g_mean_mag',
    'bp_magnitude':      'phot_bp_mean_mag',
    'rp_magnitude':      'phot_rp_mean_mag',
    'color':             'bp_rp',
    'colour':            'bp_rp',
    'temperature':       'teff_gspphot',
    'teff':              'teff_gspphot',
    'logg':              'logg_gspphot',
    'metallicity':       'mh_gspphot',
    'proper_motion_ra':  'pmra',
    'proper_motion_dec': 'pmdec',
    'variability':       'phot_variable_flag',
}

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are an astronomy query parser for the Gaia DR3 database.

Convert the user query into a single JSON object. Return ONLY valid JSON — no markdown, no explanation.

JSON schema:
{
  "intent":     string,
  "ra":         float or null,
  "dec":        float or null,
  "radius":     float or null,
  "columns":    [list of strings],
  "filters":    {dict of column: value or {min, max}},
  "join_table": string or null,
  "limit":      int
}

Supported intents:
  cone_search          — stars in a circular sky region (ra, dec, radius required)
  color_histogram      — colour distribution in a region (ra, dec, radius required)
  hr_diagram           — HR diagram; cone optional for sky-wide (parallax+G+bp_rp always added)
  stellar_population   — filter by star type; cone optional for sky-wide queries
  variability_search   — variable stars in a region (ra, dec, radius required)
  internal_crossmatch  — join with another DR3 table; cone optional (join_table required)
  nearest_neighbours   — closest stars to the Sun, sky-wide (ra/dec/radius may be null)
  velocity_computation — 3D kinematics: proper motion + radial velocity (ra/dec/radius optional)

Valid column names:
  ra, dec, source_id, parallax, parallax_error,
  pmra, pmdec, radial_velocity,
  phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, bp_rp,
  teff_gspphot, logg_gspphot, mh_gspphot,
  phot_variable_flag, ruwe

Valid join_table values (internal_crossmatch only):
  astrophysical_parameters, vari_classifier_result, vari_rrlyrae,
  vari_cepheid, vari_eclipsing_binary, vari_rotation_modulation

Rules:
  - If a target name is given (e.g. 'Pleiades', 'Andromeda'), infer ra/dec from your knowledge.
  - If coordinates are unknown and intent requires them, use galactic centre: ra=266.4, dec=-29.0
  - Default radius: 1.0 degree
  - Default limit: 1000
  - ra/dec/radius may be null for nearest_neighbours, velocity_computation, hr_diagram, stellar_population, internal_crossmatch
  - ra/dec/radius are REQUIRED for cone_search, color_histogram, variability_search
  - join_table is null unless intent is internal_crossmatch
  - filters: use {"min": x, "max": y} for ranges, or a single value for equality
  - "brightest", "faintest", "hottest", "coolest", "fastest", "nearest" — 
  these are sky-wide superlatives, never use cone_search for them.
  Use stellar_population (brightness/temperature) or nearest_neighbours (distance).

Examples:

Query: "Show stars around the Pleiades"
{"intent":"cone_search","ra":56.75,"dec":24.12,"radius":1.0,
 "columns":["source_id","ra","dec","phot_g_mean_mag"],
 "filters":{},"join_table":null,"limit":1000}

Query: "HR diagram of stars near Omega Centauri"
{"intent":"hr_diagram","ra":201.7,"dec":-47.5,"radius":1.0,
 "columns":["source_id","parallax","phot_g_mean_mag","bp_rp"],
 "filters":{"parallax":{"min":0.1}},"join_table":null,"limit":2000}

Query: "Show me the 50 nearest stars"
{"intent":"nearest_neighbours","ra":null,"dec":null,"radius":null,
 "columns":["source_id","ra","dec","parallax","phot_g_mean_mag"],
 "filters":{},"join_table":null,"limit":50}

Query: "Red dwarfs near Barnard's star"
{"intent":"stellar_population","ra":269.45,"dec":4.69,"radius":2.0,
 "columns":["source_id","ra","dec","teff_gspphot","logg_gspphot","bp_rp"],
 "filters":{"teff_gspphot":{"min":2400,"max":3900},"logg_gspphot":{"min":4.5}},
 "join_table":null,"limit":1000}

Query: "Variable stars near the galactic centre"
{"intent":"variability_search","ra":266.4,"dec":-29.0,"radius":1.0,
 "columns":["source_id","ra","dec","phot_g_mean_mag","phot_variable_flag"],
 "filters":{"phot_variable_flag":"VARIABLE"},"join_table":null,"limit":1000}

Query: "How fast do stars near Orion move?"
{"intent":"velocity_computation","ra":83.8,"dec":-5.4,"radius":2.0,
 "columns":["source_id","ra","dec","pmra","pmdec","radial_velocity","parallax"],
 "filters":{"radial_velocity":{"min":-500,"max":500}},"join_table":null,"limit":1000}

Query: "Colour histogram of stars near Andromeda"
{"intent":"color_histogram","ra":10.68,"dec":41.27,"radius":0.5,
 "columns":["source_id","bp_rp","phot_g_mean_mag","phot_bp_mean_mag","phot_rp_mean_mag"],
 "filters":{"bp_rp":{"min":-1.0,"max":6.0}},"join_table":null,"limit":1000}

Query: "Show me the hottest stars in the whole sky"
{"intent":"stellar_population","ra":null,"dec":null,"radius":null,
 "columns":["source_id","ra","dec","teff_gspphot","phot_g_mean_mag"],
 "filters":{"teff_gspphot":{"min":30000}},"join_table":null,"limit":1000}

Query: "Cross-match stars near the LMC with astrophysical parameters"
{"intent":"internal_crossmatch","ra":80.9,"dec":-69.8,"radius":2.0,
 "columns":["source_id","ra","dec","teff_gspphot","logg_gspphot"],
 "filters":{},"join_table":"astrophysical_parameters","limit":1000}

Query: "Show me the brightest stars in the sky"
{"intent":"stellar_population","ra":null,"dec":null,"radius":null,
 "columns":["source_id","ra","dec","phot_g_mean_mag","bp_rp","parallax"],
 "filters":{"phot_g_mean_mag":{"max":4.0}},
 "join_table":null,"limit":1000}
"""


def _build_prompt(user_query: str) -> str:
    return (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{user_query}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )


def parse_query(user_query: str) -> dict:
    """Send query to LLM and return parsed JSON dict."""
    prompt  = _build_prompt(user_query)
    outputs = llm.generate([prompt], SAMPLING_PARAMS)
    raw     = outputs[0].outputs[0].text.strip()
    raw = re.sub(r'^```json\s*', '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'^```\s*',     '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'\s*```$',     '', raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError as exc:
        raise ValueError(f'Model returned non-JSON:\n{raw}') from exc


In [6]:
#complex vs easy query. 

ROUTER_SYSTEM_PROMPT = """
You are a router for an astronomy query pipeline. Your job is to decide whether a user query can be answered with a SINGLE Gaia database query, or whether it requires MULTIPLE queries.

Return ONLY valid JSON, no markdown, no explanation:
{
  "complexity": "simple" | "complex",
  "reason": "<one short sentence>"
}

A query is SIMPLE if it requests one analysis, one region, one population. The analysis may have many filters or columns — that does not make it complex.

A query is COMPLEX only if it explicitly asks for multiple distinct outputs, multiple regions to compare, or sequential operations where one result feeds another.

PREFER "simple" WHEN IN DOUBT. Most queries are simple. Only mark complex when the user clearly asks for two or more separate things.

Examples:

Query: "Show me stars around the Pleiades"
{"complexity": "simple", "reason": "single cone search"}

Query: "HR diagram of nearby red dwarfs"
{"complexity": "simple", "reason": "one HR diagram with a population filter"}

Query: "Show me the brightest variable stars near the galactic centre"
{"complexity": "simple", "reason": "one filtered population in one region"}

Query: "Cross-match stars near the LMC with the Cepheid table"
{"complexity": "simple", "reason": "single crossmatch query"}

Query: "Show me the HR diagram AND the colour histogram of the Pleiades"
{"complexity": "complex", "reason": "two distinct analyses of one region"}

Query: "Compare colour distributions near Andromeda and the LMC"
{"complexity": "complex", "reason": "same analysis in two different regions"}

Query: "Find variable stars near Orion and compute their kinematics"
{"complexity": "complex", "reason": "two analyses chained on the same population"}

Query: "Show me red dwarfs and also white dwarfs"
{"complexity": "complex", "reason": "two distinct populations requested"}

Query: "What's the temperature distribution of metal-poor giants near M31"
{"complexity": "simple", "reason": "one distribution of one filtered population"}

Query: "Show me the nearest 50 stars and their HR diagram"
{"complexity": "complex", "reason": "two analyses (list + diagram) of the same selection"}
"""  

def _build_router_prompt(user_query: str) -> str:
    return (
        '<|im_start|>system\n'
        f'{ROUTER_SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{user_query}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )

def route_query(user_query: str) -> dict:
    """Classify a query as simple or complex. Returns {'complexity': ..., 'reason': ...}."""
    prompt = _build_router_prompt(user_query)
    outputs = llm.generate([prompt], SAMPLING_PARAMS)
    raw = outputs[0].outputs[0].text.strip()
    raw = re.sub(r'^```json\s*', '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'^```\s*', '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'\s*```$', '', raw)
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        print(f'  Router returned non-JSON, defaulting to simple:\n{raw}')
        return {'complexity': 'simple', 'reason': 'router fallback'}
    
    # Validate
    if result.get('complexity') not in ('simple', 'complex'):
        print(f"  Router returned invalid complexity, defaulting to simple")
        return {'complexity': 'simple', 'reason': 'router fallback'}
    
    return result


def routed_pipeline(user_query: str):
    """Top-level entry: route, then dispatch to the right sub-pipeline."""
    print(f"\n{'='*60}")
    print(f'  USER QUERY: {user_query}')
    print(f"{'='*60}\n")
    
    print('[Router] Classifying query …')
    routing = route_query(user_query)
    print(f"  → {routing['complexity'].upper()}: {routing['reason']}\n")
    
    if routing['complexity'] == 'simple':
        return supervised_pipeline(user_query)
    else:
        print('  ⚠️  Complex queries not yet supported. Falling through to simple parser anyway.')
        print('  (Phase 2: implement complex parser + orchestrator)')
        return supervised_pipeline(user_query)


## 5 · ADQL builder

In [7]:
def _resolve_columns(cols: list) -> list:
    """Resolve aliases and deduplicate column list."""
    resolved = [_COLUMN_ALIASES.get(c, c) for c in cols]
    seen, out = set(), []
    for c in resolved:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out


def _cone_clause(ra, dec, radius, ra_col='ra', dec_col='dec') -> str:
    return f"1=CONTAINS(POINT('ICRS', {ra_col}, {dec_col}), CIRCLE('ICRS', {ra}, {dec}, {radius}))"


def _filter_clauses(filters: dict) -> list:
    """
    Convert filters dict to list of ADQL WHERE fragments.
    Supports:
      {"col": value}               -> col = value  (or col = 'value' for strings)
      {"col": {"min": x}}          -> col >= x
      {"col": {"max": y}}          -> col <= y
      {"col": {"min": x, "max": y}} -> col BETWEEN x AND y
    """
    clauses = []
    for col, val in filters.items():
        col = _COLUMN_ALIASES.get(col, col)
        if isinstance(val, dict):
            mn, mx = val.get('min'), val.get('max')
            if mn is not None and mx is not None:
                clauses.append(f'{col} BETWEEN {mn} AND {mx}')
            elif mn is not None:
                clauses.append(f'{col} >= {mn}')
            elif mx is not None:
                clauses.append(f'{col} <= {mx}')
        elif isinstance(val, str):
            clauses.append(f"{col} = '{val}'")
        else:
            clauses.append(f'{col} = {val}')
    return clauses


def _null_guards(columns: list) -> list:
    """
    For GSP-Phot columns, inject IS NOT NULL guards
    so the query doesn't silently return rows with missing data.
    """
    gspphot = {'teff_gspphot', 'logg_gspphot', 'mh_gspphot'}
    return [f'{c} IS NOT NULL' for c in columns if c in gspphot]


def build_adql(q: dict) -> str:
    """
    Convert a validated parsed JSON dict into an ADQL query string.

    Intents:
      cone_search         — spatial cone, LLM-chosen columns
      color_histogram     — cone (required), forces all 4 photometry columns
      hr_diagram          — cone optional, forces parallax + G + bp_rp, adds abs mag
      stellar_population  — cone optional, teff/logg/mh filters + null guards
      variability_search  — cone (required), phot_variable_flag = 'VARIABLE'
      internal_crossmatch — cone optional, JOIN on source_id to another DR3 table
      nearest_neighbours  — sky-wide, ORDER BY parallax DESC
      velocity_computation— radial_velocity IS NOT NULL, cone optional
    """
    intent     = q['intent']
    ra         = q.get('ra')
    dec        = q.get('dec')
    radius     = q.get('radius', 1.0)
    limit      = q.get('limit', 1000)
    filters    = q.get('filters') or {}
    join_table = q.get('join_table')
    columns    = _resolve_columns(q.get('columns') or ['source_id', 'ra', 'dec'])

    # These are must include fields. 
    for col in ('source_id', 'ra', 'dec', 'phot_g_mean_mag', 'bp_rp', 'pmra', 'pmdec'):
        if col not in columns:
            columns.append(col)

    # ── Sky-wide flag (no cone given) ─────────────────────────────────────────
    sky_wide = ra is None or dec is None

    filter_parts = _filter_clauses(filters)

    # cone_search
    if intent == 'cone_search':
        cols  = ', '.join(columns)
        where = ' AND '.join([_cone_clause(ra, dec, radius)] + filter_parts)
        return f'SELECT TOP {limit} {cols} FROM gaiadr3.gaia_source WHERE {where} ORDER BY random_index'

    # color_histogram
    elif intent == 'color_histogram':
        forced = ['source_id', 'bp_rp', 'phot_g_mean_mag',
                  'phot_bp_mean_mag', 'phot_rp_mean_mag']
        cols   = ', '.join(_resolve_columns(forced + columns))
        where_parts = [
            _cone_clause(ra, dec, radius),
            'bp_rp IS NOT NULL',
        ] + filter_parts
        where = ' AND '.join(where_parts)
        return f'SELECT TOP {limit} {cols} FROM gaiadr3.gaia_source WHERE {where} ORDER BY random_index'

    # hr_diagram
    elif intent == 'hr_diagram':
        forced  = ['source_id', 'parallax', 'phot_g_mean_mag', 'bp_rp']
        cols    = _resolve_columns(forced + columns)
        select_cols  = ', '.join(cols)
        select_cols += ', (phot_g_mean_mag + 5*LOG10(parallax/100)) AS abs_g_mag'
        where_parts  = ['parallax > 0', 'bp_rp IS NOT NULL'] + filter_parts
        if not sky_wide:
            where_parts.insert(0, _cone_clause(ra, dec, radius))
        where = ' AND '.join(where_parts)
        return f'SELECT TOP {limit} {select_cols} FROM gaiadr3.gaia_source WHERE {where} ORDER BY random_index'

    # stellar_population
    elif intent == 'stellar_population':
        cols        = ', '.join(columns)
        null_guards = _null_guards(columns)
        where_parts = null_guards + filter_parts
        if not sky_wide:
            where_parts.insert(0, _cone_clause(ra, dec, radius))
        where = ' AND '.join(where_parts) if where_parts else '1=1'
        return f'SELECT TOP {limit} {cols} FROM gaiadr3.gaia_source WHERE {where} ORDER BY random_index'

    # variability_search
    elif intent == 'variability_search':
        if 'phot_variable_flag' not in columns:
            columns.append('phot_variable_flag')
        cols        = ', '.join(columns)
        where_parts = [
            _cone_clause(ra, dec, radius),
            "phot_variable_flag = 'VARIABLE'",
        ] + [fp for fp in filter_parts
             if 'phot_variable_flag' not in fp]
        where = ' AND '.join(where_parts)
        return f'SELECT TOP {limit} {cols} FROM gaiadr3.gaia_source WHERE {where} ORDER BY random_index'

    # internal_crossmatch 
    elif intent == 'internal_crossmatch':
        g_cols = ', '.join(f'g.{c}' if c in VALID_COLUMNS else c
                           for c in columns)
        where_parts = filter_parts[:]
        if not sky_wide:
            where_parts.insert(0, _cone_clause(ra, dec, radius, 'g.ra', 'g.dec'))
        where = ' AND '.join(where_parts) if where_parts else '1=1'
        return (
            f'SELECT TOP {limit} {g_cols} '
            f'FROM gaiadr3.gaia_source AS g '
            f'JOIN gaiadr3.{join_table} AS x ON g.source_id = x.source_id '
            f'WHERE {where} ORDER BY random_index'
        )

    # nearest_neighbours
    elif intent == 'nearest_neighbours':
        if 'parallax' not in columns:
            columns.insert(0, 'parallax')
        cols        = ', '.join(columns)
        where_parts = ['parallax > 0'] + filter_parts
        where       = ' AND '.join(where_parts)
        return (
            f'SELECT TOP {limit} {cols} '
            f'FROM gaiadr3.gaia_source '
            f'WHERE {where} '
            f'ORDER BY parallax DESC'
        )

    # velocity_computation 
    elif intent == 'velocity_computation':
        forced = ['source_id', 'ra', 'dec', 'pmra', 'pmdec', 'radial_velocity']
        cols   = ', '.join(_resolve_columns(forced + columns))
        where_parts = ['radial_velocity IS NOT NULL'] + filter_parts
        if not sky_wide:
            where_parts.insert(0, _cone_clause(ra, dec, radius))
        where = ' AND '.join(where_parts)
        return f'SELECT TOP {limit} {cols} FROM gaiadr3.gaia_source WHERE {where} ORDER BY random_index'

    else:
        raise ValueError(f"Unknown intent: '{intent}'")

## 6 · Gaia TAP query runner

In [8]:
def run_query(adql: str, retries: int = 3, backoff: float = 15.0, row_limit: int = 50_000) -> pd.DataFrame:
    """
    Submit ADQL to the Gaia TAP service, return a pandas DataFrame.
    Injects TOP N if missing. Fails fast on HTTP 500 (bad syntax).
    """
    from astroquery.gaia import Gaia

    Gaia.MAIN_GAIA_TABLE = 'gaiadr3.gaia_source'
    Gaia.ROW_LIMIT       = row_limit

    if not re.search(r'\bTOP\s+\d+\b', adql, re.IGNORECASE):
        adql = re.sub(r'\bSELECT\b', f'SELECT TOP {row_limit}',
                      adql, count=1, flags=re.IGNORECASE)
        print(f'Injected TOP {row_limit}')

    if not re.search(r'\bWHERE\b', adql, re.IGNORECASE):
        print('⚠️  No WHERE clause — query may time out.')

    print(f'\nQuery sent:\n{adql}\n')

    last_exc = None
    for attempt in range(retries):
        try:
            job    = Gaia.launch_job_async(adql, verbose=False)
            result = job.get_results()
            print(f'Got {len(result):,} rows')
            return result.to_pandas()
        except Exception as exc:
            last_exc = exc
            print(f'[attempt {attempt+1}/{retries}] {exc}')
            if '500' in str(exc):
                print('HTTP 500 — bad query syntax, not retrying.')
                raise exc
            if attempt < retries - 1:
                wait = backoff * (2 ** attempt)
                print(f'Waiting {wait:.0f}s …')
                time.sleep(wait)
    raise last_exc

## 7 · Teacher — JSON & ADQL validator

In [9]:
MAX_RETRIES = 3

# Intents where cone is fully optional (sky-wide queries allowed)
_CONE_OPTIONAL_INTENTS = {
    'nearest_neighbours', 'velocity_computation',
    'hr_diagram', 'stellar_population', 'internal_crossmatch',
}
# Intents that always require a spatial anchor
_CONE_REQUIRED_INTENTS = {'cone_search', 'color_histogram', 'variability_search'}


def validate_parsed_json(q: dict) -> list:
    """Return list of error strings. Empty list means valid."""
    errors = []
    intent = q.get('intent')

    # ── intent ────────────────────────────────────────────────────────────────
    if not intent or intent not in VALID_INTENTS:
        errors.append(
            f"'intent' is '{intent}', must be one of: {sorted(VALID_INTENTS)}."
        )
        return errors  # no point checking the rest

    # ── spatial fields ────────────────────────────────────────────────────────
    needs_cone = intent in _CONE_REQUIRED_INTENTS
    for field in ('ra', 'dec'):
        val = q.get(field)
        if val is None:
            if needs_cone:
                errors.append(
                    f"'{field}' is null for intent '{intent}'. "
                    f"Provide coordinates or use galactic centre (ra=266.4, dec=-29.0)."
                )
        else:
            if not isinstance(val, (int, float)):
                errors.append(f"'{field}' must be a number, got: {repr(val)}.")
            elif field == 'ra' and not (0 <= float(val) <= 360):
                errors.append(f"'ra' = {val} is outside [0, 360].")
            elif field == 'dec' and not (-90 <= float(val) <= 90):
                errors.append(f"'dec' = {val} is outside [-90, 90].")

    radius = q.get('radius')
    if needs_cone:
        if radius is None:
            errors.append("'radius' is null. Use default 1.0 degree.")
        elif not isinstance(radius, (int, float)) or float(radius) <= 0:
            errors.append(f"'radius' must be positive, got: {repr(radius)}.")
        elif float(radius) > 10:
            errors.append(f"'radius' = {radius}° is very large (> 10°) and may time out.")

    # ── limit ─────────────────────────────────────────────────────────────────
    limit = q.get('limit', 1000)
    if not isinstance(limit, int) or limit <= 0:
        errors.append(f"'limit' must be a positive integer, got: {repr(limit)}.")
    elif limit > 100_000:
        errors.append(f"'limit' = {limit:,} is too large. Use ≤ 100,000.")

    # ── columns ───────────────────────────────────────────────────────────────
    cols    = q.get('columns') or []
    unknown = [c for c in cols
               if c not in VALID_COLUMNS and c not in _COLUMN_ALIASES]
    if unknown:
        errors.append(
            f"Unknown column(s): {unknown}. "
            f"Allowed: {sorted(VALID_COLUMNS)}."
        )

    # ── join_table ────────────────────────────────────────────────────────────
    join_table = q.get('join_table')
    if intent == 'internal_crossmatch':
        if not join_table:
            errors.append(
                f"'join_table' is required for internal_crossmatch. "
                f"Choose one of: {sorted(VALID_JOIN_TABLES)}."
            )
        elif join_table not in VALID_JOIN_TABLES:
            errors.append(
                f"'join_table' = '{join_table}' is not a valid DR3 table. "
                f"Choose one of: {sorted(VALID_JOIN_TABLES)}."
            )
    elif join_table is not None:
        errors.append(
            f"'join_table' should be null for intent '{intent}', got: '{join_table}'."
        )

    return errors


def validate_adql(adql: str) -> list:
    """Catch common problems in the final ADQL string."""
    errors = []
    if re.search(r'\bNone\b', adql):
        errors.append(
            "ADQL contains literal 'None' — a coordinate was not resolved. "
            "Replace with numeric degree values."
        )
    if not re.search(r'\bTOP\s+\d+\b', adql, re.IGNORECASE):
        errors.append('No TOP clause — add TOP N to cap result size.')
    if not re.search(r'\bWHERE\b', adql, re.IGNORECASE):
        errors.append('No WHERE clause — add a spatial or quality filter.')
    return errors


def build_correction_prompt(original_query: str, bad_json: dict,
                             errors: list, attempt: int) -> str:
    error_block = '\n'.join(f'  - {e}' for e in errors)
    return (
        f'Your previous answer for the query:\n'
        f'  "{original_query}"\n\n'
        f'produced this JSON:\n'
        f'{json.dumps(bad_json, indent=2)}\n\n'
        f'The teacher found {len(errors)} error(s) (attempt {attempt}/{MAX_RETRIES}):\n'
        f'{error_block}\n\n'
        f'Fix ALL errors and return ONLY corrected JSON. No markdown, no explanation.'
    )


print('Validator defined.')

Validator defined.


## 8 · Cost judge

In [10]:
COST_JUDGE_SYSTEM_PROMPT = """You are an expert in ADQL queries on the Gaia DR3 database (1.8 billion rows).
Evaluate how expensive a query will be on the Gaia TAP server.

Return ONLY valid JSON, no markdown:
{
  "verdict":       "cheap" | "moderate" | "expensive" | "dangerous",
  "score":         0-100,
  "reasons":       [list of strings],
  "optimisations": [list of concrete ADQL rewrites],
  "safe_to_run":   true | false
}

Scoring guide:
   0-25  cheap     : TOP ≤ 1000, tight cone ≤ 0.5°, indexed filter, no JOIN
  26-50  moderate  : TOP ≤ 10000, cone ≤ 2°, simple filters
  51-75  expensive : TOP ≤ 50000, cone > 2°, ORDER BY non-indexed col, weak parallax
  76-100 dangerous : no TOP or TOP > 100000, no spatial filter, full-table scan, multi-JOIN

Key cost drivers:
  - Missing/large TOP (biggest risk)
  - Cone radius > 2 degrees
  - ORDER BY on parallax or radial_velocity (not indexed)
  - radial_velocity IS NOT NULL (only ~1% of stars, forces full scan)
  - parallax > 0  (matches ~90% of rows — very weak)
  - JOIN without a tight spatial filter
  - No WHERE clause

Optimisation strategies:
  - Reduce TOP N
  - Shrink cone radius
  - Replace parallax > 0 with parallax > 10 (nearby stars)
  - Add phot_g_mean_mag < 15 to cut faint stars early
  - Remove ORDER BY or replace with a parallax threshold filter
  - For sky-wide queries with no CONTAINS clause: suggest MOD(random_index, N) = 0
    as a uniform sky sample instead of adding a spatial cone.
    N=100 for moderate, N=1000 for expensive, N=10000 for dangerous."""


def _build_cost_prompt(adql: str) -> str:
    return (
        '<|im_start|>system\n'
        f'{COST_JUDGE_SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'Evaluate this ADQL query:\n\n{adql}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )


def fast_cost_precheck(adql: str):
    """Rule-based fast check — no LLM call. Returns dict if dangerous, else None."""
    adql_upper = adql.upper()
    if 'CONTAINS' not in adql_upper and 'WHERE' not in adql_upper:
        return {
            'verdict': 'dangerous', 'score': 100, 'safe_to_run': False,
            'reasons': ['No WHERE clause — full scan of 1.8B rows.'],
            'optimisations': ['Add MOD(random_index, 10000) = 0 for a uniform sky sample.'],
        }
    top_match = re.search(r'\bTOP\s+(\d+)\b', adql, re.IGNORECASE)
    if not top_match:
        return {
            'verdict': 'dangerous', 'score': 95, 'safe_to_run': False,
            'reasons': ['No TOP clause — unbounded result set.'],
            'optimisations': ['Add TOP 1000 after SELECT.'],
        }
    if int(top_match.group(1)) > 100_000:
        return {
            'verdict': 'dangerous', 'score': 90, 'safe_to_run': False,
            'reasons': [f'TOP {int(top_match.group(1)):,} exceeds 100,000 row limit.'],
            'optimisations': ['Reduce TOP to ≤ 10,000.'],
        }
    return None


def evaluate_query_cost(adql: str) -> dict:
    """LLM judge: score and explain query cost."""
    prompt  = _build_cost_prompt(adql)
    outputs = llm.generate([prompt], JUDGE_SAMPLING_PARAMS)
    raw     = outputs[0].outputs[0].text.strip()
    raw = re.sub(r'^```json\s*', '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'^```\s*',     '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'\s*```$',     '', raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f'Cost judge returned non-JSON:\n{raw}')
        return {
            'verdict': 'expensive', 'score': 70, 'safe_to_run': False,
            'reasons': ['Cost judge failed to parse — defaulting conservative.'],
            'optimisations': [],
        }


def auto_optimise_adql(adql: str, optimisations: list,
                       top_cap: int = 5_000, score: int = 0) -> str:
    """Apply deterministic optimisations to the ADQL string."""

    #Inject random sampling for sky-wide queries
    has_cone     = 'CONTAINS' in adql.upper()
    has_sampling = 'random_index' in adql.lower()
    if not has_cone and not has_sampling:
        if score >= 75:
            sample_n = 10000   # 0.01% — dangerous
        elif score >= 50:
            sample_n = 1000    # 0.1%  — expensive
        else:
            sample_n = 100     # 1%    — moderate
        adql = re.sub(
            r'\bWHERE\b',
            f'WHERE MOD(random_index, {sample_n}) = 0 AND',
            adql, count=1, flags=re.IGNORECASE
        )
        print(f'  Injected MOD(random_index, {sample_n}) — uniform sky sample')

    #Cap Top M 
    def cap_top(m):
        n = int(m.group(1))
        if n > top_cap:
            print(f'  TOP {n:,} → {top_cap:,}')
            return f'TOP {top_cap}'
        return m.group(0)
    adql = re.sub(r'\bTOP\s+(\d+)\b', cap_top, adql, flags=re.IGNORECASE)

    # ── Shrink large cones ────────────────────────────────────────────────────
    def shrink_cone(m):
        if float(m.group(3)) > 3.0:
            print(f'  Cone {m.group(3)}° → 2.0°')
            return f"CIRCLE('ICRS', {m.group(1)}, {m.group(2)}, 2.0)"
        return m.group(0)
    adql = re.sub(
        r"CIRCLE\s*\(\s*'ICRS'\s*,\s*([0-9.\-]+)\s*,\s*([0-9.\-]+)\s*,\s*([0-9.]+)\s*\)",
        shrink_cone, adql, flags=re.IGNORECASE
    )

    # ── Tighten weak parallax filter ──────────────────────────────────────────
    before = adql
    adql   = re.sub(r'\bparallax\s*>\s*0\b', 'parallax > 1', adql, flags=re.IGNORECASE)
    if adql != before:
        print('  parallax > 0  →  parallax > 1')

    if optimisations:
        print('  LLM judge also suggested:')
        for opt in optimisations:
            print(f'    • {opt}')
    return adql


_VERDICT_EMOJI = {'cheap': '🟢', 'moderate': '🟡', 'expensive': '🟠', 'dangerous': '🔴'}


def print_cost_report(cost: dict):
    verdict = cost.get('verdict', '?')
    score   = cost.get('score', '?')
    emoji   = _VERDICT_EMOJI.get(verdict, '⚪')
    print(f"  {'─'*52}")
    print(f"  {emoji}  COST: {verdict.upper()}  (score {score}/100)")
    print(f"  {'─'*52}")
    for r in cost.get('reasons', []):
        print(f'    ⚠️  {r}')
    for o in cost.get('optimisations', []):
        print(f'    💡 {o}')
    print(f"  {'─'*52}")


## 9 · Supervised pipeline

In [32]:
import os as _os_mod
import json as _json_mod
import time as _time_mod

OUTPUT_DIR = './pipeline_outputs'
_os_mod.makedirs(OUTPUT_DIR, exist_ok=True)


def supervised_pipeline(user_query: str, source_id_filter: list = None) -> pd.DataFrame:
    """
    Full pipeline:
      1. LLM parses query → JSON
      2. Teacher validates JSON  (correction loop)
      3. build_adql()            (+ source_id_filter injection if supplied)
      4. Teacher validates ADQL  (correction loop)
      5. Cost judge              (block / optimise / approve)
      6. run_query()             (retry on transient errors)

    Returns a DataFrame and writes a JSON record to OUTPUT_DIR,
    matching the output format of complex_pipeline.
    Raises RuntimeError after MAX_RETRIES exhausted.
    """
    print(f"\n{'='*60}")
    print(f'  USER QUERY: {user_query}')
    if source_id_filter:
        print(f'  SOURCE_ID FILTER: {len(source_id_filter)} IDs supplied')
    print(f"{'='*60}\n")

    current_prompt = user_query
    parsed         = None
    adql           = None
    t0             = _time_mod.time()

    for attempt in range(1, MAX_RETRIES + 1):

        # 1 — LLM → JSON
        print(f'[Attempt {attempt}/{MAX_RETRIES}] Parsing …')
        try:
            parsed = parse_query(current_prompt)
        except ValueError as exc:
            print(f'  Non-JSON from LLM: {exc}')
            current_prompt = (
                f'Your answer was not valid JSON.\n'
                f'Original query: "{user_query}"\n'
                f'Error: {exc}\n'
                f'Return ONLY a valid JSON object.'
            )
            continue
        print(f'  Parsed JSON:\n{json.dumps(parsed, indent=2)}')

        # 2 — validate JSON
        json_errors = validate_parsed_json(parsed)
        if json_errors:
            print(f'  JSON errors ({len(json_errors)}):')
            for e in json_errors:
                print(f'    • {e}')
            current_prompt = build_correction_prompt(
                user_query, parsed, json_errors, attempt
            )
            continue
        print('  JSON valid.')

        # 3 — build ADQL
        try:
            adql = build_adql(parsed)
        except (ValueError, KeyError) as exc:
            print(f'  build_adql() failed: {exc}')
            current_prompt = build_correction_prompt(
                user_query, parsed, [f'build_adql() error: {exc}'], attempt
            )
            continue

        # 3b — inject source_id filter if supplied
        if source_id_filter:
            ids  = ', '.join(str(int(i)) for i in source_id_filter[:200])
            adql = re.sub(r'\bWHERE\b',
                          f'WHERE source_id IN ({ids}) AND',
                          adql, count=1, flags=re.IGNORECASE)
            print(f'  ↳ Injected source_id filter '
                  f'({min(len(source_id_filter), 200)} IDs'
                  f'{" capped" if len(source_id_filter) > 200 else ""})')

        print(f'  ADQL: {adql}')

        # 4 — validate ADQL
        adql_errors = validate_adql(adql)
        if adql_errors:
            print(f'  ADQL errors ({len(adql_errors)}):')
            for e in adql_errors:
                print(f'    • {e}')
            current_prompt = build_correction_prompt(
                user_query, parsed, adql_errors, attempt
            )
            continue
        print('  ADQL syntax valid.')

        # 5 — cost gate
        print('  Evaluating cost …')
        cost = fast_cost_precheck(adql)
        if cost is None:
            cost = evaluate_query_cost(adql)
        print_cost_report(cost)

        if cost['verdict'] == 'dangerous':
            print('  🔴 BLOCKED — query too dangerous.')
            current_prompt = build_correction_prompt(
                user_query, parsed,
                [f"Query rated DANGEROUS (score {cost['score']}/100). "
                 f"Reasons: {'; '.join(cost['reasons'])}. "
                 f"Fix: {'; '.join(cost['optimisations'])}"],
                attempt,
            )
            continue

        if not cost['safe_to_run']:
            adql = auto_optimise_adql(
                adql, cost.get('optimisations', []), score=cost.get('score', 0)
            )
            print(f'  Optimised ADQL: {adql}')

        # 6 — execute
        try:
            df       = run_query(adql)
            duration = round(_time_mod.time() - t0, 2)

            # ── Build output record (same schema as complex_pipeline) ─────
            output = {
                'query':         user_query,
                'routed_as':     'simple',
                'composition':   'simple',
                'outer_attempt': 1,
                'plan': {
                    'intent':        parsed.get('intent'),
                    'ra':            parsed.get('ra'),
                    'dec':           parsed.get('dec'),
                    'radius':        parsed.get('radius'),
                    'filters':       parsed.get('filters', {}),
                    'join_table':    parsed.get('join_table'),
                    'limit':         parsed.get('limit'),
                    'source_id_filter_size': len(source_id_filter) if source_id_filter else 0,
                },
                'execution': [{
                    'step_id':          1,
                    'intent':           parsed.get('intent'),
                    'status':           'success',
                    'adql':             adql,
                    'rows_returned':    len(df),
                    'duration_seconds': duration,
                    'error':            None,
                }],
                'summary': {
                    'total_steps':            1,
                    'successful':             1,
                    'failed':                 0,
                    'skipped':                0,
                    'total_duration_seconds': duration,
                },
            }

            safe_name = re.sub(r'[^a-z0-9]+', '_', user_query.lower())[:60]
            out_path  = _os_mod.path.join(OUTPUT_DIR, f'{safe_name}.json')
            with open(out_path, 'w') as fh:
                _json_mod.dump(output, fh, indent=2)

            print(f'\n  ✓  {len(df)} rows in {duration}s')
            print(f'  Output JSON → {out_path}')
            return {'output_json': output, 'results': {1: df}}

        except Exception as exc:
            print(f'  run_query() failed: {exc}')
            if '500' in str(exc):
                current_prompt = build_correction_prompt(
                    user_query, parsed,
                    [f'ADQL caused HTTP 500 on Gaia TAP: {exc}'],
                    attempt,
                )
                continue
            raise

    raise RuntimeError(
        f'Pipeline failed after {MAX_RETRIES} attempts for query: "{user_query}"'
    )


<H1> COMPLEX PART: 

In [12]:
# ═══════════════════════════════════════════════════════════════════════════
#  COMPLEX PIPELINE — four notebook cells
#
#  Paste each labeled block into its own cell in this order:
#
#   CELL A  →  new cell after section 9.5 (Routed pipeline)
#              markdown header: ## 10 · Decomposer
#
#   CELL B  →  new cell after CELL A
#              markdown header: ## 11 · Plan validator
#
#   CELL C  →  new cell after CELL B
#              markdown header: ## 12 · Orchestrator
#
#   CELL D  →  new cell after CELL C
#              markdown header: ## 13 · Complex pipeline (top-level)
#
#  Then update the run cell to call complex_pipeline() directly for testing,
#  or let routed_pipeline() dispatch to it automatically.
# ═══════════════════════════════════════════════════════════════════════════


# ┌─────────────────────────────────────────────────────────────────────────┐
# │  CELL A · Decomposer                                                    │
# └─────────────────────────────────────────────────────────────────────────┘

# Increase tokens for the decomposer — plans are longer than single-intent JSON
DECOMPOSER_SAMPLING_PARAMS = SamplingParams(temperature=0.0, max_tokens=1024)

DECOMPOSER_SYSTEM_PROMPT = f"""You are a query decomposer for an astronomy pipeline that queries the Gaia DR3 database.

A complex user query has already been identified as requiring multiple analyses.
Your job is to choose the right composition type and produce the minimal plan.

Return ONLY valid JSON — no markdown, no explanation.

COMPOSITION TYPES — choose exactly one:

  "merged"     — TWO OR MORE analyses on the SAME region that can be answered
                 by ONE ADQL query. This is the most efficient option.
                 Use when all intents are from this mergeable set:
                   cone_search, color_histogram, hr_diagram, stellar_population,
                   variability_search, velocity_computation
                 A merged plan has NO 'steps'. It has 'merged_intents' instead.
                 ALWAYS prefer merged over parallel when the region is the same.

  "parallel"   — N INDEPENDENT queries that CANNOT be merged because:
                   • they cover DIFFERENT sky regions, OR
                   • they use internal_crossmatch or nearest_neighbours
                     (these require a JOIN or special ORDER BY — not mergeable)
                 A parallel plan has a 'steps' list with no depends_on.

  "sequential" — N DEPENDENT queries where step N needs source_ids from step N-1.
                 Use when the query chains operations (find → crossmatch → analyse).
                 A sequential plan has a 'steps' list where some steps have depends_on.

OUTPUT SCHEMA by composition:

Merged:
{{
  "shared_region": {{"target": string or null, "ra": float, "dec": float, "radius": float}},
  "composition":   "merged",
  "merged_intents": [list of intent strings],
  "columns":       [additional columns beyond each intent's defaults],
  "filters":       {{dict of column: value or {{min, max}}}},
  "limit":         int
}}

Parallel / Sequential:
{{
  "shared_region": {{"target": string or null, "ra": float, "dec": float, "radius": float}} or null,
  "composition":   "parallel" | "sequential",
  "steps": [
    {{
      "step_id":                int,
      "intent":                 string,
      "description":            string,
      "natural_language_query": string,
      "depends_on":             [list of step_id ints],
      "dependency_type":        "source_id_filter" or null,
      "override_region":        {{"ra": float, "dec": float, "radius": float}} or null,
      "filters":                {{dict}},
      "columns":                [list of strings],
      "join_table":             string or null,
      "limit":                  int
    }}
  ]
}}

FIELD RULES:

  shared_region   — required for merged. For parallel/sequential: hoist here when
                    steps share the same region; null when regions differ or sky-wide.
  merged_intents  — present ONLY when composition = "merged". List of 2+ intent names.
  steps           — present ONLY when composition = "parallel" or "sequential".
  composition     — "merged" is preferred over "parallel" when the region is the same
                    and all intents are in the mergeable set.
  depends_on      — [] for parallel steps. Sequential downstream steps list upstream step_id(s).
  dependency_type — null when depends_on is empty; "source_id_filter" otherwise.
  override_region — set only when a parallel step needs a DIFFERENT region from shared_region.
  limit           — merged: default 2000. Steps with downstream dependants: ≤ 200.

Supported intents:
  cone_search          — stars in a circular sky region (ra/dec/radius required)
  color_histogram      — colour distribution in a region (ra/dec/radius required)
  hr_diagram           — HR diagram; cone optional (parallax+G+bp_rp always added)
  stellar_population   — filter by star type; cone optional for sky-wide queries
  variability_search   — variable stars in a region (ra/dec/radius required)
  internal_crossmatch  — join with another DR3 table (join_table required) — NOT mergeable
  nearest_neighbours   — closest stars to the Sun, sky-wide — NOT mergeable
  velocity_computation — 3D kinematics: pm + radial_velocity (cone optional)

Valid join_table values (internal_crossmatch only):
  {', '.join(sorted(VALID_JOIN_TABLES))}

Valid column names:
  {', '.join(sorted(VALID_COLUMNS))}

Rules:
  - PREFER merged when two or more analyses share the same region.
  - Decompose into the MINIMUM number of steps/intents.
  - If a target name is given (e.g. "Pleiades"), infer ra/dec from your knowledge.
    Unknown targets: use galactic centre ra=266.4, dec=-29.0.
  - Default radius: 1.0 degree.
  - Steps with downstream dependants MUST include source_id in their columns.

Examples:

Query: "Show me the HR diagram and the colour histogram of the Pleiades"
→ Same region, both intents mergeable → composition = "merged"
{{
  "shared_region": {{"target": "Pleiades", "ra": 56.75, "dec": 24.12, "radius": 1.0}},
  "composition": "merged",
  "merged_intents": ["hr_diagram", "color_histogram"],
  "columns": [],
  "filters": {{}},
  "limit": 2000
}}

Query: "Show me the HR diagram and velocity of stars near Omega Centauri"
→ Same region, both intents mergeable → composition = "merged"
{{
  "shared_region": {{"target": "Omega Centauri", "ra": 201.7, "dec": -47.5, "radius": 1.0}},
  "composition": "merged",
  "merged_intents": ["hr_diagram", "velocity_computation"],
  "columns": [],
  "filters": {{}},
  "limit": 2000
}}

Query: "Compare colour distributions near Andromeda and the LMC"
→ DIFFERENT regions → composition = "parallel" (cannot merge different regions)
{{
  "shared_region": null,
  "composition": "parallel",
  "steps": [
    {{
      "step_id": 1, "intent": "color_histogram",
      "description": "Colour histogram near Andromeda",
      "natural_language_query": "Colour histogram of stars near Andromeda",
      "depends_on": [], "dependency_type": null,
      "override_region": {{"ra": 10.68, "dec": 41.27, "radius": 0.5}},
      "filters": {{}}, "columns": ["source_id","bp_rp","phot_g_mean_mag"],
      "join_table": null, "limit": 2000
    }},
    {{
      "step_id": 2, "intent": "color_histogram",
      "description": "Colour histogram near the LMC",
      "natural_language_query": "Colour histogram of stars near the LMC",
      "depends_on": [], "dependency_type": null,
      "override_region": {{"ra": 80.9, "dec": -69.8, "radius": 2.0}},
      "filters": {{}}, "columns": ["source_id","bp_rp","phot_g_mean_mag"],
      "join_table": null, "limit": 2000
    }}
  ]
}}

Query: "Show me red dwarfs and also white dwarfs"
→ Sky-wide, incompatible filters (different teff/logg ranges) → composition = "parallel"
{{
  "shared_region": null,
  "composition": "parallel",
  "steps": [
    {{
      "step_id": 1, "intent": "stellar_population",
      "description": "Red dwarfs sky-wide",
      "natural_language_query": "Show me red dwarfs",
      "depends_on": [], "dependency_type": null, "override_region": null,
      "filters": {{"teff_gspphot": {{"min": 2400, "max": 3900}}, "logg_gspphot": {{"min": 4.5}}}},
      "columns": ["source_id","ra","dec","teff_gspphot","logg_gspphot","phot_g_mean_mag","bp_rp"],
      "join_table": null, "limit": 1000
    }},
    {{
      "step_id": 2, "intent": "stellar_population",
      "description": "White dwarfs sky-wide",
      "natural_language_query": "Show me white dwarfs",
      "depends_on": [], "dependency_type": null, "override_region": null,
      "filters": {{"teff_gspphot": {{"min": 5000}}, "logg_gspphot": {{"min": 7.0}}}},
      "columns": ["source_id","ra","dec","teff_gspphot","logg_gspphot","phot_g_mean_mag","bp_rp"],
      "join_table": null, "limit": 1000
    }}
  ]
}}

Query: "Find variable stars near Orion, cross-match with the Cepheid table, and plot their HR diagram"
→ Sequential dependency chain → composition = "sequential"
{{
  "shared_region": {{"target": "Orion", "ra": 83.82, "dec": -5.39, "radius": 2.0}},
  "composition": "sequential",
  "steps": [
    {{
      "step_id": 1, "intent": "variability_search",
      "description": "Variable stars near Orion",
      "natural_language_query": "Variable stars near Orion",
      "depends_on": [], "dependency_type": null, "override_region": null,
      "filters": {{"phot_variable_flag": "VARIABLE"}},
      "columns": ["source_id","ra","dec","phot_g_mean_mag","phot_variable_flag"],
      "join_table": null, "limit": 200
    }},
    {{
      "step_id": 2, "intent": "internal_crossmatch",
      "description": "Cross-match with Cepheid table",
      "natural_language_query": "Cross-match stars near Orion with the Cepheid table",
      "depends_on": [1], "dependency_type": "source_id_filter", "override_region": null,
      "filters": {{}},
      "columns": ["source_id","ra","dec","phot_g_mean_mag","bp_rp"],
      "join_table": "vari_cepheid", "limit": 200
    }},
    {{
      "step_id": 3, "intent": "hr_diagram",
      "description": "HR diagram of matched Cepheids",
      "natural_language_query": "HR diagram of Cepheid variable stars near Orion",
      "depends_on": [2], "dependency_type": "source_id_filter", "override_region": null,
      "filters": {{}},
      "columns": ["source_id","parallax","phot_g_mean_mag","bp_rp"],
      "join_table": null, "limit": 1000
    }}
  ]
}}
"""



def _build_decomposer_prompt(user_query: str) -> str:
    return (
        '<|im_start|>system\n'
        f'{DECOMPOSER_SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{user_query}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )


def _build_decomposer_correction_prompt(original_query: str,
                                        bad_plan: dict,
                                        errors: list,
                                        attempt: int) -> str:
    error_list = '\n'.join(f'  - {e}' for e in errors)
    return (
        f'Your previous plan for the query "{original_query}" had errors '
        f'(attempt {attempt}/{MAX_RETRIES}):\n'
        f'{error_list}\n\n'
        f'Bad plan:\n{json.dumps(bad_plan, indent=2)}\n\n'
        f'Fix all errors and return ONLY valid JSON matching the required schema.'
    )


def decompose_query(user_query: str) -> dict:
    """
    Call the LLM to break a complex query into a structured plan.
    Returns a plan dict. Raises RuntimeError after MAX_RETRIES failures.
    """
    current_prompt = user_query
    last_plan = {}

    for attempt in range(1, MAX_RETRIES + 1):
        print(f'  [Decomposer attempt {attempt}/{MAX_RETRIES}] …')

        prompt  = _build_decomposer_prompt(current_prompt)
        outputs = llm.generate([prompt], DECOMPOSER_SAMPLING_PARAMS)
        raw     = outputs[0].outputs[0].text.strip()
        raw = re.sub(r'^```json\s*', '', raw, flags=re.IGNORECASE)
        raw = re.sub(r'^```\s*',     '', raw, flags=re.IGNORECASE)
        raw = re.sub(r'\s*```$',     '', raw)

        try:
            plan = json.loads(raw)
        except json.JSONDecodeError:
            print(f'    Non-JSON from decomposer:\n{raw[:200]}')
            current_prompt = (
                f'Your answer for "{user_query}" was not valid JSON.\n'
                f'Return ONLY a valid JSON object matching the required schema.'
            )
            last_plan = {}
            continue

        last_plan = plan
        errors    = validate_plan(plan)

        if not errors:
            composition = plan['composition']
            if composition == 'merged':
                n = len(plan.get('merged_intents', []))
                print(f'    Plan valid. {n} merged intents, composition=merged')
            else:
                n = len(plan.get('steps', []))
                print(f'    Plan valid. {n} steps, composition={composition}')
            return plan

        print(f'    Plan errors ({len(errors)}):')
        for e in errors:
            print(f'      • {e}')
        current_prompt = _build_decomposer_correction_prompt(
            user_query, plan, errors, attempt
        )

    raise RuntimeError(
        f'Decomposer failed after {MAX_RETRIES} attempts for: "{user_query}"\n'
        f'Last plan:\n{json.dumps(last_plan, indent=2)}'
    )


print('Decomposer defined.')


# ┌─────────────────────────────────────────────────────────────────────────┐
# │  CELL B · Plan validator                                                │
# └─────────────────────────────────────────────────────────────────────────┘

# Intents that require a spatial cone (shared_region or override_region)
_CONE_REQUIRED_INTENTS = {'cone_search', 'color_histogram', 'variability_search'}


def validate_plan(plan: dict) -> list:
    """
    Deterministic validation of a decomposer plan.
    Returns a list of error strings; empty list means the plan is valid.
    Mirrors validate_parsed_json in structure and style.
    """
    errors = []

    # ── 1. Top-level structure ────────────────────────────────────────────
    if not isinstance(plan, dict):
        return ['Plan must be a JSON object.']

    composition = plan.get('composition')
    if composition not in ('parallel', 'sequential', 'merged'):
        errors.append(
            f"'composition' must be 'parallel', 'sequential', or 'merged', "
            f"got: {composition!r}"
        )

    # Merged plans have merged_intents instead of steps
    if composition == 'merged':
        merged_intents = plan.get('merged_intents')
        if not isinstance(merged_intents, list) or len(merged_intents) < 2:
            errors.append(
                "'merged_intents' must be a list of at least 2 intent strings."
            )
        shared_region = plan.get('shared_region')
        if not shared_region:
            errors.append("merged plan requires 'shared_region' with ra/dec/radius.")
        else:
            for field in ('ra', 'dec', 'radius'):
                if shared_region.get(field) is None:
                    errors.append(f"shared_region.{field} must not be null.")
        if plan.get('steps') is not None:
            errors.append(
                "merged plan must not have a 'steps' field — "
                "use 'merged_intents' instead."
            )
        return errors   # no step-level or DAG checks needed for merged

    steps = plan.get('steps')
    if not isinstance(steps, list) or len(steps) == 0:
        errors.append("'steps' must be a non-empty list.")
        return errors

    shared_region = plan.get('shared_region')
    if shared_region is not None:
        for field in ('ra', 'dec', 'radius'):
            if shared_region.get(field) is None:
                errors.append(f"shared_region.{field} must not be null.")

    # ── 2. Per-step checks ────────────────────────────────────────────────
    seen_ids = set()
    for step in steps:
        sid = step.get('step_id')

        # step_id
        if not isinstance(sid, int):
            errors.append(f"step_id must be an integer, got: {sid!r}")
        elif sid in seen_ids:
            errors.append(f"Duplicate step_id: {sid}")
        else:
            seen_ids.add(sid)

        # intent
        intent = step.get('intent')
        if intent not in VALID_INTENTS:
            errors.append(
                f"Step {sid}: intent {intent!r} is not valid. "
                f"Choose from: {sorted(VALID_INTENTS)}"
            )

        # natural_language_query
        nlq = step.get('natural_language_query', '')
        if not isinstance(nlq, str) or not nlq.strip():
            errors.append(f"Step {sid}: 'natural_language_query' must be a non-empty string.")

        # description
        if not isinstance(step.get('description', ''), str):
            errors.append(f"Step {sid}: 'description' must be a string.")

        # depends_on / dependency_type consistency
        depends_on      = step.get('depends_on', [])
        dependency_type = step.get('dependency_type')

        if not isinstance(depends_on, list):
            errors.append(f"Step {sid}: 'depends_on' must be a list.")
            depends_on = []

        if depends_on and dependency_type != 'source_id_filter':
            errors.append(
                f"Step {sid}: 'dependency_type' must be 'source_id_filter' "
                f"when depends_on is non-empty, got: {dependency_type!r}"
            )
        if not depends_on and dependency_type is not None:
            errors.append(
                f"Step {sid}: 'dependency_type' must be null when depends_on is empty."
            )

        # join_table
        join_table = step.get('join_table')
        if intent == 'internal_crossmatch':
            if not join_table:
                errors.append(
                    f"Step {sid}: 'join_table' is required for internal_crossmatch. "
                    f"Choose from: {sorted(VALID_JOIN_TABLES)}"
                )
            elif join_table not in VALID_JOIN_TABLES:
                errors.append(
                    f"Step {sid}: join_table {join_table!r} is not a valid DR3 table."
                )
        elif join_table is not None:
            errors.append(
                f"Step {sid}: 'join_table' must be null for intent {intent!r}."
            )

        # cone-required intents need a region
        if intent in _CONE_REQUIRED_INTENTS:
            has_shared   = shared_region is not None
            has_override = step.get('override_region') is not None
            if not has_shared and not has_override:
                errors.append(
                    f"Step {sid}: intent {intent!r} requires a spatial region. "
                    f"Set shared_region or override_region."
                )

        # override_region structure, if present
        override = step.get('override_region')
        if override is not None:
            for field in ('ra', 'dec', 'radius'):
                if override.get(field) is None:
                    errors.append(
                        f"Step {sid}: override_region.{field} must not be null."
                    )

        # limit
        limit = step.get('limit', 1000)
        if not isinstance(limit, int) or limit <= 0:
            errors.append(f"Step {sid}: 'limit' must be a positive integer.")

        # upstream steps that have dependants must include source_id
        # (checked after building the full step list below)

    # ── 3. DAG checks ─────────────────────────────────────────────────────
    # All referenced step_ids must exist
    for step in steps:
        for dep_id in step.get('depends_on', []):
            if dep_id not in seen_ids:
                errors.append(
                    f"Step {step['step_id']}: depends_on references "
                    f"unknown step_id {dep_id}."
                )

    # Cycle detection via topological sort (Kahn's algorithm)
    adj      = {s['step_id']: set(s.get('depends_on', [])) for s in steps}
    in_degree = {sid: 0 for sid in seen_ids}
    for sid, deps in adj.items():
        for d in deps:
            if d in in_degree:
                in_degree[sid] += 1
    queue    = [sid for sid, deg in in_degree.items() if deg == 0]
    visited  = 0
    while queue:
        node = queue.pop(0)
        visited += 1
        for sid in seen_ids:
            if node in adj.get(sid, set()):
                in_degree[sid] -= 1
                if in_degree[sid] == 0:
                    queue.append(sid)
    if visited != len(seen_ids):
        errors.append('Cycle detected in step dependencies.')

    # composition vs depends_on consistency
    all_empty = all(len(s.get('depends_on', [])) == 0 for s in steps)
    any_set   = any(len(s.get('depends_on', [])) > 0 for s in steps)
    if composition == 'parallel' and not all_empty:
        errors.append(
            "composition is 'parallel' but some steps have non-empty depends_on."
        )
    if composition == 'sequential' and not any_set:
        errors.append(
            "composition is 'sequential' but no steps have depends_on set."
        )

    # ── 4. Upstream steps with dependants must expose source_id ──────────
    steps_with_dependants = set()
    for step in steps:
        for dep_id in step.get('depends_on', []):
            steps_with_dependants.add(dep_id)

    step_map = {s['step_id']: s for s in steps}
    for sid in steps_with_dependants:
        if sid not in step_map:
            continue   # already flagged above
        upstream_cols = step_map[sid].get('columns', [])
        if 'source_id' not in upstream_cols:
            errors.append(
                f"Step {sid} is referenced by downstream steps but does not "
                f"include 'source_id' in its columns."
            )

    return errors


print('Plan validator defined.')


# ┌─────────────────────────────────────────────────────────────────────────┐
# │  CELL C · Merged ADQL builder                                           │
# │                                                                         │
# │  build_adql_merged() produces ONE ADQL string from a list of intents    │
# │  that share the same region and base table (gaiadr3.gaia_source).       │
# │                                                                         │
# │  The output contract for the entire pipeline is:                        │
# │    simple path  → supervised_pipeline  → one ADQL → one DataFrame       │
# │    merged path  → build_adql_merged    → one ADQL → one DataFrame       │
# │    parallel path → N independent ADQL  → N DataFrames                  │
# │    sequential path → N dependent ADQL  → N DataFrames (source_id chain) │
# └─────────────────────────────────────────────────────────────────────────┘

# Per-intent metadata used by build_adql_merged.
# Each entry defines:
#   forced_cols  — columns always added regardless of what the user asked for
#   where_guards — WHERE predicates always injected for this intent
#   select_extra — computed SELECT expressions (no alias conflict across intents)
#   incompatible — intents this one cannot share a query with
#                  (internal_crossmatch needs a JOIN; nearest_neighbours needs
#                   ORDER BY parallax DESC — both break the shared query model)

_INTENT_META = {
    'cone_search': {
        'forced_cols':  ['source_id', 'ra', 'dec'],
        'where_guards': [],
        'select_extra': [],
        'incompatible': {'internal_crossmatch', 'nearest_neighbours'},
    },
    'color_histogram': {
        'forced_cols':  ['source_id', 'bp_rp', 'phot_g_mean_mag',
                         'phot_bp_mean_mag', 'phot_rp_mean_mag'],
        'where_guards': ['bp_rp IS NOT NULL'],
        'select_extra': [],
        'incompatible': {'internal_crossmatch', 'nearest_neighbours'},
    },
    'hr_diagram': {
        'forced_cols':  ['source_id', 'parallax', 'phot_g_mean_mag', 'bp_rp'],
        'where_guards': ['parallax > 0', 'bp_rp IS NOT NULL'],
        'select_extra': ['(phot_g_mean_mag + 5*LOG10(parallax/100)) AS abs_g_mag'],
        'incompatible': {'internal_crossmatch', 'nearest_neighbours'},
    },
    'stellar_population': {
        'forced_cols':  ['source_id', 'ra', 'dec'],
        'where_guards': [],          # null guards added dynamically from columns
        'select_extra': [],
        'incompatible': {'internal_crossmatch', 'nearest_neighbours'},
    },
    'variability_search': {
        'forced_cols':  ['source_id', 'ra', 'dec', 'phot_variable_flag'],
        'where_guards': ["phot_variable_flag = 'VARIABLE'"],
        'select_extra': [],
        'incompatible': {'internal_crossmatch', 'nearest_neighbours'},
    },
    'velocity_computation': {
        'forced_cols':  ['source_id', 'ra', 'dec', 'pmra', 'pmdec', 'radial_velocity'],
        'where_guards': ['radial_velocity IS NOT NULL'],
        'select_extra': [],
        'incompatible': {'internal_crossmatch', 'nearest_neighbours'},
    },
    # nearest_neighbours and internal_crossmatch are NOT mergeable —
    # they require structural changes to the query (ORDER BY parallax DESC
    # and JOIN respectively) that break the shared-query model.
    # The decomposer is instructed never to emit them as merged.
}

_MERGEABLE_INTENTS = set(_INTENT_META.keys())


def build_adql_merged(plan: dict) -> str:
    """
    Build one ADQL query from a merged plan.

    Merging rules:
      - SELECT: union of forced_cols across all intents + user columns + computed exprs
      - WHERE:  cone clause (required for merged) + union of all intent guards + user filters
      - FROM:   always gaiadr3.gaia_source (no JOINs in merged queries)
      - ORDER:  random_index (compatible with all mergeable intents)
      - LIMIT:  plan['limit'], default 2000

    Raises ValueError if any intent is not mergeable.
    """
    intents = plan.get('merged_intents', [])
    if not intents:
        raise ValueError("build_adql_merged: 'merged_intents' is empty.")

    for intent in intents:
        if intent not in _MERGEABLE_INTENTS:
            raise ValueError(
                f"build_adql_merged: intent '{intent}' is not mergeable. "
                f"Mergeable intents: {sorted(_MERGEABLE_INTENTS)}"
            )

    # Check incompatibility pairs
    intent_set = set(intents)
    for intent in intents:
        bad = _INTENT_META[intent]['incompatible'] & intent_set
        if bad:
            raise ValueError(
                f"build_adql_merged: intent '{intent}' is incompatible with {bad}."
            )

    # Spatial region — required for merged queries
    region = plan.get('shared_region') or {}
    ra     = region.get('ra')
    dec    = region.get('dec')
    radius = region.get('radius', 1.0)
    if ra is None or dec is None:
        raise ValueError(
            "build_adql_merged: shared_region with ra/dec is required for merged queries."
        )

    limit   = plan.get('limit', 2000)
    filters = plan.get('filters') or {}

    # ── Build column list ────────────────────────────────────────────────
    # Start with the universal base columns, then add per-intent forced cols,
    # then any user-specified columns from the plan.
    base_cols = ['source_id', 'ra', 'dec', 'phot_g_mean_mag', 'bp_rp', 'pmra', 'pmdec']
    all_cols  = list(base_cols)

    for intent in intents:
        for col in _INTENT_META[intent]['forced_cols']:
            if col not in all_cols:
                all_cols.append(col)

    for col in _resolve_columns(plan.get('columns') or []):
        if col not in all_cols:
            all_cols.append(col)

    # Computed SELECT expressions (e.g. abs_g_mag for hr_diagram)
    # Deduplicate by alias name so two hr_diagram-like intents don't double-add
    seen_aliases = set()
    select_extras = []
    for intent in intents:
        for expr in _INTENT_META[intent]['select_extra']:
            alias = expr.split(' AS ')[-1].strip() if ' AS ' in expr else expr
            if alias not in seen_aliases:
                seen_aliases.add(alias)
                select_extras.append(expr)

    select_cols = ', '.join(all_cols)
    if select_extras:
        select_cols += ', ' + ', '.join(select_extras)

    # ── Build WHERE clause ────────────────────────────────────────────────
    where_parts = [_cone_clause(ra, dec, radius)]

    # Per-intent guards — deduplicate
    seen_guards = set()
    for intent in intents:
        for guard in _INTENT_META[intent]['where_guards']:
            if guard not in seen_guards:
                seen_guards.add(guard)
                where_parts.append(guard)

    # Null guards for any GSP-Phot columns in the SELECT list
    for guard in _null_guards(all_cols):
        if guard not in where_parts:
            where_parts.append(guard)

    # User filters
    where_parts.extend(_filter_clauses(filters))

    where = ' AND '.join(where_parts)

    return (
        f'SELECT TOP {limit} {select_cols} '
        f'FROM gaiadr3.gaia_source '
        f'WHERE {where} '
        f'ORDER BY random_index'
    )


def validate_merged_plan(plan: dict) -> list:
    """
    Additional validation for merged plans, on top of validate_plan.
    Returns list of error strings.
    """
    errors = []
    intents = plan.get('merged_intents', [])

    if not intents:
        errors.append("merged plan must have a non-empty 'merged_intents' list.")
        return errors

    for intent in intents:
        if intent not in _MERGEABLE_INTENTS:
            errors.append(
                f"Intent '{intent}' cannot be merged. "
                f"Mergeable: {sorted(_MERGEABLE_INTENTS)}"
            )

    intent_set = set(intents)
    for intent in intents:
        bad = _INTENT_META.get(intent, {}).get('incompatible', set()) & intent_set
        if bad:
            errors.append(f"Intent '{intent}' is incompatible with {bad} in a merged query.")

    if not plan.get('shared_region'):
        errors.append("merged plan requires 'shared_region' with ra/dec/radius.")
    else:
        for field in ('ra', 'dec', 'radius'):
            if plan['shared_region'].get(field) is None:
                errors.append(f"shared_region.{field} must not be null for merged plan.")

    return errors


print('Merged ADQL builder defined.')


# ┌─────────────────────────────────────────────────────────────────────────┐
# │  CELL D · Orchestrator                                                  │
# │                                                                         │
# │  Three execution paths, all producing ADQL:                             │
# │    merged     — build_adql_merged → cost judge → run_query              │
# │    parallel   — N × supervised_pipeline (each produces its own ADQL)    │
# │    sequential — N × supervised_pipeline with source_id injection        │
# │                                                                         │
# │  The cost judge (fast_cost_precheck / evaluate_query_cost) fires on     │
# │  every ADQL string regardless of path — same function, same thresholds. │
# └─────────────────────────────────────────────────────────────────────────┘

STEP_RETRIES = 2


def _effective_query(step: dict, plan: dict) -> str:
    """Natural-language query for supervised_pipeline, with coordinates if override."""
    nlq      = step['natural_language_query']
    override = step.get('override_region')
    if override:
        nlq = (
            f'{nlq} '
            f'(ra={override["ra"]}, dec={override["dec"]}, '
            f'radius={override["radius"]} deg)'
        )
    return nlq


def _build_step_correction_prompt(original_query: str,
                                  step: dict,
                                  error: str,
                                  attempt: int) -> str:
    return (
        f'The following Gaia DR3 query failed (attempt {attempt}/{STEP_RETRIES}):\n'
        f'  Original step query: "{original_query}"\n'
        f'  Intent: {step["intent"]}\n'
        f'  Error: {error}\n\n'
        f'Rewrite the query to fix the error. '
        f'Keep the same intent ({step["intent"]}) and spatial region. '
        f'Return only the corrected natural-language query string.'
    )


def _execute_step(step: dict, plan: dict, source_id_filter: list = None) -> tuple:
    """
    Execute one plan step through supervised_pipeline with a per-step retry loop.
    Returns (DataFrame | None, timing_dict) where timing_dict includes the ADQL.
    Cost judge fires inside supervised_pipeline — same as the simple path.
    """
    base_query    = _effective_query(step, plan)
    current_query = base_query
    last_error    = None
    t0            = time.time()

    for attempt in range(1, STEP_RETRIES + 1):
        if attempt > 1:
            print(f'    [Step retry {attempt}/{STEP_RETRIES}]')
            current_query = _build_step_correction_prompt(
                base_query, step, str(last_error), attempt
            )
        t0 = time.time()
        try:
            df       = supervised_pipeline(current_query,
                                           source_id_filter=source_id_filter)
            duration = round(time.time() - t0, 2)
            return df, {'status': 'success', 'rows': len(df), 'duration': duration}

        except RuntimeError as exc:
            last_error = exc
            duration   = round(time.time() - t0, 2)
            if 'DANGEROUS' in str(exc) or 'dangerous' in str(exc).lower():
                print(f'  🔴  Step {step["step_id"]} blocked by cost judge.')
                return None, {'status': 'blocked', 'error': str(exc), 'duration': duration}
            print(f'  ✗  Step {step["step_id"]} failed (attempt {attempt}): {exc}')

        except Exception as exc:
            last_error = exc
            print(f'  ✗  Step {step["step_id"]} unexpected error (attempt {attempt}): {exc}')

    return None, {'status': 'failed', 'error': str(last_error),
                  'duration': round(time.time() - t0, 2)}


def _topological_order(steps: list) -> list:
    id_to_step = {s['step_id']: s for s in steps}
    in_degree  = {s['step_id']: len(s.get('depends_on', [])) for s in steps}
    queue      = sorted([sid for sid, deg in in_degree.items() if deg == 0])
    order      = []
    while queue:
        sid = queue.pop(0)
        order.append(id_to_step[sid])
        for step in steps:
            if sid in step.get('depends_on', []):
                in_degree[step['step_id']] -= 1
                if in_degree[step['step_id']] == 0:
                    queue.append(step['step_id'])
                    queue.sort()
    return order


def run_merged(plan: dict) -> tuple:
    """
    Merged path: build one ADQL, run cost judge, execute once.
    Returns (results: dict, timings: dict, failed_ids: list).

    results = {'merged': DataFrame | None}
    timings = {'merged': {status, adql, rows, duration}}
    """
    print(f'\n[Merged] Intents: {plan["merged_intents"]}')
    t0 = time.time()

    # Build the single ADQL
    try:
        adql = build_adql_merged(plan)
    except ValueError as exc:
        timing = {'status': 'failed', 'error': str(exc), 'duration': 0, 'adql': None}
        return {'merged': None}, {'merged': timing}, ['merged']
    print(f'  ADQL: {adql}')

    # Cost judge — same path as supervised_pipeline
    print('  Evaluating cost …')
    cost = fast_cost_precheck(adql)
    if cost is None:
        cost = evaluate_query_cost(adql)
    print_cost_report(cost)

    if cost['verdict'] == 'dangerous':
        print('  🔴  Blocked by cost judge.')
        timing = {'status': 'blocked', 'adql': adql,
                  'error': f"Dangerous query (score {cost['score']})",
                  'duration': round(time.time() - t0, 2)}
        return {'merged': None}, {'merged': timing}, ['merged']

    if not cost['safe_to_run']:
        adql = auto_optimise_adql(adql, cost.get('optimisations', []),
                                  score=cost.get('score', 0))
        print(f'  Optimised ADQL: {adql}')

    # Execute
    try:
        df       = run_query(adql)
        duration = round(time.time() - t0, 2)
        timing   = {'status': 'success', 'adql': adql,
                    'rows': len(df), 'duration': duration}
        print(f'  ✓  {len(df)} rows in {duration}s')
        return {'merged': df}, {'merged': timing}, []
    except Exception as exc:
        duration = round(time.time() - t0, 2)
        timing   = {'status': 'failed', 'adql': adql,
                    'error': str(exc), 'duration': duration}
        print(f'  ✗  {exc}')
        return {'merged': None}, {'merged': timing}, ['merged']


def run_parallel(plan: dict) -> tuple:
    """N independent steps, each through supervised_pipeline."""
    results, timings, failed_ids = {}, {}, []
    for step in plan['steps']:
        sid = step['step_id']
        print(f'\n[Step {sid}] {step["description"]}')
        print(f'  Query: {_effective_query(step, plan)}')
        df, timing   = _execute_step(step, plan)
        results[sid] = df
        timings[sid] = timing
        if timing['status'] == 'success':
            print(f'  ✓  {timing["rows"]} rows in {timing["duration"]}s')
        else:
            failed_ids.append(sid)
    return results, timings, failed_ids


def run_sequential(plan: dict) -> tuple:
    """Steps in topological order with source_id injection."""
    results, timings, failed_ids = {}, {}, []
    for step in _topological_order(plan['steps']):
        sid        = step['step_id']
        depends_on = step.get('depends_on', [])
        print(f'\n[Step {sid}] {step["description"]}')
        print(f'  Query: {_effective_query(step, plan)}')

        # Skip if any upstream failed
        skip_reason = None
        for dep_id in depends_on:
            if timings.get(dep_id, {}).get('status') != 'success':
                skip_reason = f'upstream step {dep_id} did not succeed'
                break
            if 'source_id' not in (results.get(dep_id) or pd.DataFrame()).columns:
                skip_reason = f'upstream step {dep_id} has no source_id column'
                break
        if skip_reason:
            print(f'  ⊘  Skipping: {skip_reason}')
            results[sid] = None
            timings[sid] = {'status': 'skipped', 'reason': skip_reason, 'duration': 0}
            failed_ids.append(sid)
            continue

        # Collect upstream source_ids
        source_ids = None
        if depends_on:
            ids = []
            for dep_id in depends_on:
                ids.extend(results[dep_id]['source_id'].dropna().astype(int).tolist())
            source_ids = list(dict.fromkeys(ids))
            print(f'  ↳  {len(source_ids)} source_ids from step(s) {depends_on}')

        df, timing   = _execute_step(step, plan, source_id_filter=source_ids)
        results[sid] = df
        timings[sid] = timing
        if timing['status'] == 'success':
            print(f'  ✓  {timing["rows"]} rows in {timing["duration"]}s')
        else:
            failed_ids.append(sid)

    return results, timings, failed_ids


def orchestrate(plan: dict) -> tuple:
    """
    Dispatch to the right execution path based on composition.
    Returns (results, timings, failed_ids).
    """
    composition = plan['composition']
    n_steps     = len(plan.get('steps') or plan.get('merged_intents', []))
    print(f"\n{'─'*60}")
    print(f'  ORCHESTRATOR  {composition.upper()}  {n_steps} intent(s)/step(s)')
    print(f"{'─'*60}")

    if composition == 'merged':
        return run_merged(plan)
    elif composition == 'parallel':
        return run_parallel(plan)
    else:
        return run_sequential(plan)


print('Orchestrator defined.')


# ┌─────────────────────────────────────────────────────────────────────────┐
# │  CELL E · Complex pipeline (top-level)                                  │
# │                                                                         │
# │  Output contract:                                                        │
# │    merged     → output_json contains one  'adql' string                 │
# │    parallel   → output_json contains a    list of 'adql' strings        │
# │    sequential → output_json contains a    list of 'adql' strings        │
# └─────────────────────────────────────────────────────────────────────────┘

import json as _json_mod
import os   as _os_mod

OUTPUT_DIR   = './pipeline_outputs'
PLAN_RETRIES = 2
_os_mod.makedirs(OUTPUT_DIR, exist_ok=True)


def _build_plan_correction_prompt(user_query, plan, failed_ids, timings, attempt):
    lines = [
        f'The plan for "{user_query}" had {len(failed_ids)} failing step(s) '
        f'(outer attempt {attempt}/{PLAN_RETRIES}).',
        '', 'Failed steps/keys:',
    ]
    for sid in failed_ids:
        t = timings.get(sid, {})
        lines.append(
            f'  {sid} — {t.get("status","unknown")}: '
            f'{t.get("error") or t.get("reason","unknown")}'
        )
    lines += ['', 'Previous plan:', json.dumps(plan, indent=2), '',
              'Revise to fix failing steps/intents. Keep successful ones unchanged. '
              'Return ONLY valid JSON matching the required schema.']
    return '\n'.join(lines)


def build_output_json(user_query, plan, timings, outer_attempt):
    """
    Assemble the structured output record.
    The 'adql' field(s) record every ADQL string that was generated,
    so the output is transparent about exactly what hit the TAP server.
    """
    composition = plan['composition']

    if composition == 'merged':
        t = timings.get('merged', {})
        execution = [{
            'key':              'merged',
            'intents':          plan.get('merged_intents', []),
            'status':           t.get('status', 'unknown'),
            'adql':             t.get('adql'),
            'rows_returned':    t.get('rows', 0),
            'duration_seconds': t.get('duration', 0),
            'error':            t.get('error'),
        }]
        successful = 1 if t.get('status') == 'success' else 0
        failed     = 0 if successful else 1
        skipped    = 0
    else:
        execution = []
        for step in plan['steps']:
            sid = step['step_id']
            t   = timings.get(sid, {})
            execution.append({
                'step_id':          sid,
                'intent':           step['intent'],
                'description':      step['description'],
                'status':           t.get('status', 'unknown'),
                'adql':             t.get('adql'),        # populated if supervised_pipeline exposes it
                'rows_returned':    t.get('rows', 0),
                'duration_seconds': t.get('duration', 0),
                'error':            t.get('error') or t.get('reason'),
            })
        successful = sum(1 for e in execution if e['status'] == 'success')
        failed     = sum(1 for e in execution if e['status'] in ('failed', 'blocked'))
        skipped    = sum(1 for e in execution if e['status'] == 'skipped')

    total_dur = round(sum(e['duration_seconds'] for e in execution), 2)

    return {
        'query':         user_query,
        'routed_as':     'complex',
        'composition':   composition,
        'outer_attempt': outer_attempt,
        'plan':          plan,
        'execution':     execution,
        'summary': {
            'total_steps':            len(execution),
            'successful':             successful,
            'failed':                 failed,
            'skipped':                skipped,
            'total_duration_seconds': total_dur,
        },
    }


def complex_pipeline(user_query: str, save_json: bool = True) -> dict:
    """
    Full complex pipeline with outer plan-level retry loop.

    Output contract (matches simple path):
      merged     → one ADQL string, one DataFrame  (results['merged'])
      parallel   → N ADQL strings, N DataFrames    (results[step_id])
      sequential → N ADQL strings, N DataFrames    (results[step_id])

    Returns:
      {
        'output_json': { query, composition, plan, execution, summary },
        'results':     { 'merged': df }  or  { step_id: df, ... }
      }
    """
    print(f"\n{'='*60}")
    print(f'  COMPLEX PIPELINE')
    print(f'  Query: {user_query}')
    print(f"{'='*60}\n")

    decompose_prompt = user_query
    last_output      = None
    last_results     = {}

    for outer_attempt in range(1, PLAN_RETRIES + 1):
        if outer_attempt > 1:
            print(f'\n[Outer retry {outer_attempt}/{PLAN_RETRIES}] Re-planning …')

        # 1. Decompose
        print(f'[{outer_attempt}.1] Decomposing …')
        try:
            plan = decompose_query(decompose_prompt)
        except RuntimeError as exc:
            raise RuntimeError(f'Decomposer failed: {exc}') from exc

        # Extra validation for merged plans
        if plan.get('composition') == 'merged':
            merge_errors = validate_merged_plan(plan)
            if merge_errors:
                raise RuntimeError(
                    f'Merged plan is invalid: {merge_errors}\n'
                    f'Plan: {json.dumps(plan, indent=2)}'
                )

        # Print plan summary
        composition = plan['composition']
        if composition == 'merged':
            print(f'\n  Plan (merged): {plan.get("merged_intents")} '
                  f'over {plan.get("shared_region", {}).get("target", "region")}\n')
        else:
            print(f'\n  Plan ({composition}, {len(plan["steps"])} step(s)):')
            for s in plan['steps']:
                dep = f' → depends on {s["depends_on"]}' if s.get('depends_on') else ''
                print(f'    Step {s["step_id"]}: {s["intent"]:25} '
                      f'{s["description"][:40]}{dep}')
            print()

        # 2. Execute
        print(f'[{outer_attempt}.2] Executing …')
        results, timings, failed_ids = orchestrate(plan)

        # 3. Record
        output       = build_output_json(user_query, plan, timings, outer_attempt)
        last_output  = output
        last_results = results
        s = output['summary']
        print(f'\n  Attempt {outer_attempt}: '
              f'✓ {s["successful"]}  ✗ {s["failed"]}  ⊘ {s["skipped"]}  '
              f'({s["total_duration_seconds"]}s)')

        if not failed_ids:
            print('  All steps succeeded.')
            break

        if outer_attempt < PLAN_RETRIES:
            print(f'  {failed_ids} failed — revising plan …')
            decompose_prompt = _build_plan_correction_prompt(
                user_query, plan, failed_ids, timings, outer_attempt
            )
        else:
            print(f'  {failed_ids} still failing — returning partial results.')

    # Write JSON
    if save_json and last_output:
        safe_name = re.sub(r'[^a-z0-9]+', '_', user_query.lower())[:60]
        out_path  = _os_mod.path.join(OUTPUT_DIR, f'{safe_name}.json')
        with open(out_path, 'w') as fh:
            _json_mod.dump(last_output, fh, indent=2)
        print(f'\n  Output JSON → {out_path}')

    s = last_output['summary']
    print(f"\n{'='*60}")
    print(f"  SUMMARY  ({last_output['composition'].upper()}, "
          f"outer attempt {last_output['outer_attempt']})")
    print(f"  Steps: {s['total_steps']}  "
          f"✓ {s['successful']}  ✗ {s['failed']}  ⊘ {s['skipped']}")
    print(f"  Time : {s['total_duration_seconds']}s")
    print(f"{'='*60}\n")

    return {'output_json': last_output, 'results': last_results}


print('Complex pipeline defined.')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Plan retries: {PLAN_RETRIES}  |  Step retries: {STEP_RETRIES}')

Decomposer defined.
Plan validator defined.
Merged ADQL builder defined.
Orchestrator defined.
Complex pipeline defined.
Output directory: ./pipeline_outputs
Plan retries: 2  |  Step retries: 2


In [13]:
# ═══════════════════════════════════════════════════════════════════════════
#  MANUAL TEST CELL — source_id_filter injection
#
#  Paste into a new notebook cell and run AFTER all existing cells are
#  loaded. Tests the modified supervised_pipeline in isolation before
#  building the orchestrator on top of it.
#
#  The test runs in three stages:
#    Stage 1 — verify build_adql injects the filter correctly (no LLM, no TAP)
#    Stage 2 — run a real upstream query to get actual source_ids from Gaia
#    Stage 3 — run a real downstream query filtered by those source_ids
# ═══════════════════════════════════════════════════════════════════════════

import re as _re


# ── Patch build_adql to accept an optional source_id_filter ─────────────────
# We wrap the existing function rather than replacing it, so the original
# behaviour is completely unchanged when source_id_filter is None.

_original_build_adql = build_adql   # keep a reference to the existing function

def build_adql(q: dict, source_id_filter: list = None) -> str:
    """
    Drop-in replacement for build_adql.
    Calls the original, then optionally injects a source_id IN (...) clause.
    """
    adql = _original_build_adql(q)

    if source_id_filter:
        ids_to_use = source_id_filter[:200]          # hard cap — TAP URL limit
        ids_str    = ', '.join(str(int(i)) for i in ids_to_use)
        # Insert right after WHERE, before any existing predicate
        adql = _re.sub(
            r'\bWHERE\b',
            f'WHERE source_id IN ({ids_str}) AND',
            adql, count=1, flags=_re.IGNORECASE,
        )
        n_capped = len(source_id_filter) - len(ids_to_use)
        cap_note = f' (capped from {len(source_id_filter)})' if n_capped else ''
        print(f'  ↳ Injected source_id filter: {len(ids_to_use)} IDs{cap_note}')

    return adql


# Also patch supervised_pipeline to accept and forward the new parameter.
_original_supervised_pipeline = supervised_pipeline

def supervised_pipeline(user_query: str,
                        source_id_filter: list = None) -> pd.DataFrame:
    """
    Drop-in replacement for supervised_pipeline.
    Accepts an optional source_id_filter that is forwarded to build_adql.
    All other behaviour (retry loop, cost judge, etc.) is unchanged.
    """
    print(f"\n{'='*60}")
    print(f'  USER QUERY: {user_query}')
    if source_id_filter:
        print(f'  SOURCE_ID FILTER: {len(source_id_filter)} IDs supplied')
    print(f"{'='*60}\n")

    current_prompt = user_query
    parsed = None
    adql   = None

    for attempt in range(1, MAX_RETRIES + 1):

        # 1 — LLM → JSON
        print(f'[Attempt {attempt}/{MAX_RETRIES}] Parsing …')
        try:
            parsed = parse_query(current_prompt)
        except ValueError as exc:
            print(f'  Non-JSON from LLM: {exc}')
            current_prompt = (
                f'Your answer was not valid JSON.\n'
                f'Original query: "{user_query}"\n'
                f'Error: {exc}\n'
                f'Return ONLY a valid JSON object.'
            )
            continue

        print(f'  Parsed JSON:\n{json.dumps(parsed, indent=2)}')

        # 2 — validate JSON
        json_errors = validate_parsed_json(parsed)
        if json_errors:
            print(f'  JSON errors ({len(json_errors)}):')
            for e in json_errors:
                print(f'    • {e}')
            current_prompt = build_correction_prompt(
                user_query, parsed, json_errors, attempt
            )
            continue
        print('  JSON valid.')

        # 3 — build ADQL (pass filter through)
        try:
            adql = build_adql(parsed, source_id_filter=source_id_filter)
        except (ValueError, KeyError) as exc:
            print(f'  build_adql() failed: {exc}')
            current_prompt = build_correction_prompt(
                user_query, parsed, [f'build_adql() error: {exc}'], attempt
            )
            continue
        print(f'  ADQL: {adql}')

        # 4 — validate ADQL
        adql_errors = validate_adql(adql)
        if adql_errors:
            print(f'  ADQL errors ({len(adql_errors)}):')
            for e in adql_errors:
                print(f'    • {e}')
            current_prompt = build_correction_prompt(
                user_query, parsed, adql_errors, attempt
            )
            continue
        print('  ADQL syntax valid.')

        # 5 — cost gate
        print('  Evaluating cost …')
        cost = fast_cost_precheck(adql)
        if cost is None:
            cost = evaluate_query_cost(adql)
        print_cost_report(cost)

        if cost['verdict'] == 'dangerous':
            print('  🔴 BLOCKED — query too dangerous.')
            current_prompt = build_correction_prompt(
                user_query, parsed,
                [f"Query rated DANGEROUS (score {cost['score']}/100). "
                 f"Reasons: {'; '.join(cost['reasons'])}. "
                 f"Fix: {'; '.join(cost['optimisations'])}"],
                attempt,
            )
            continue

        if not cost['safe_to_run']:
            adql = auto_optimise_adql(
                adql, cost.get('optimisations', []), score=cost.get('score', 0)
            )
            print(f'  Optimised ADQL: {adql}')

        # 6 — execute
        try:
            return run_query(adql)
        except Exception as exc:
            print(f'  run_query() failed: {exc}')
            if '500' in str(exc):
                current_prompt = build_correction_prompt(
                    user_query, parsed,
                    [f'ADQL caused HTTP 500 on Gaia TAP: {exc}'],
                    attempt,
                )
                continue
            raise

    raise RuntimeError(
        f'Pipeline failed after {MAX_RETRIES} attempts for query: "{user_query}"'
    )


print('Patched build_adql and supervised_pipeline with source_id_filter support.')
print()


# ════════════════════════════════════════════════════════════════════════════
#  STAGE 1 — Unit test: does build_adql inject correctly? (no LLM, no TAP)
# ════════════════════════════════════════════════════════════════════════════
print('─' * 60)
print('STAGE 1 — build_adql injection (offline, no LLM/TAP)')
print('─' * 60)

# Minimal valid parsed dict — stellar_population, sky-wide
_test_parsed = {
    'intent':  'stellar_population',
    'ra': None, 'dec': None, 'radius': 1.0,
    'columns': ['source_id', 'phot_g_mean_mag', 'teff_gspphot'],
    'filters': {'teff_gspphot': {'min': 2400, 'max': 3900}},
    'join_table': None,
    'limit': 100,
}
_test_ids = [123456789, 987654321, 111222333]

# Without filter
adql_plain = build_adql(_test_parsed)
print(f'\nWithout filter:\n  {adql_plain}')

# With filter
adql_filtered = build_adql(_test_parsed, source_id_filter=_test_ids)
print(f'\nWith 3 IDs:\n  {adql_filtered}')

# Cap test — pass 250 IDs, expect only 200 in the query
_many_ids = list(range(1_000_000, 1_000_250))
adql_capped = build_adql(_test_parsed, source_id_filter=_many_ids)
id_count_in_adql = adql_capped.count(',') + 1  # rough count inside IN(...)
# more reliable: count occurrences between IN( and the next )
import re as re2
match = re2.search(r'IN \(([^)]+)\)', adql_capped)
actual_count = len(match.group(1).split(',')) if match else -1
print(f'\nWith 250 IDs (cap=200): {actual_count} IDs in ADQL  '
      f'{"✓" if actual_count == 200 else "✗ UNEXPECTED"}')

# Verify the filter appears BEFORE the existing WHERE predicates
has_id_first = 'source_id IN' in adql_filtered.split('WHERE')[1].split('AND')[0]
print(f'\nsource_id IN appears immediately after WHERE: '
      f'{"✓" if has_id_first else "✗ UNEXPECTED"}')

print()


# ════════════════════════════════════════════════════════════════════════════
#  STAGE 2 — Upstream query: variability_search near the Pleiades
#  Goal: get real source_ids to pass downstream
# ════════════════════════════════════════════════════════════════════════════
print('─' * 60)
print('STAGE 2 — Upstream query (variable stars near Pleiades)')
print('─' * 60)

UPSTREAM_QUERY = 'Variable stars near the Pleiades'
df_upstream = supervised_pipeline(UPSTREAM_QUERY)

print(f'\nUpstream result: {len(df_upstream)} rows')
print(df_upstream[['source_id', 'phot_g_mean_mag', 'bp_rp']].head(5).to_string())

source_ids = df_upstream['source_id'].dropna().astype(int).tolist()
print(f'\nExtracted {len(source_ids)} source_ids for downstream step.')
print()


# ════════════════════════════════════════════════════════════════════════════
#  STAGE 3 — Downstream query: HR diagram filtered by upstream source_ids
#  This is the core of sequential dependency injection.
# ════════════════════════════════════════════════════════════════════════════
print('─' * 60)
print('STAGE 3 — Downstream query (HR diagram, filtered by upstream IDs)')
print('─' * 60)

DOWNSTREAM_QUERY = 'HR diagram of the matched stars'
df_downstream = supervised_pipeline(DOWNSTREAM_QUERY,
                                    source_id_filter=source_ids)

print(f'\nDownstream result: {len(df_downstream)} rows')
print(df_downstream[['source_id', 'phot_g_mean_mag', 'bp_rp', 'parallax']].head(5).to_string())

# Sanity check: every source_id in the downstream result should be in the upstream set
upstream_set   = set(source_ids)
downstream_ids = set(df_downstream['source_id'].dropna().astype(int).tolist())
stray_ids      = downstream_ids - upstream_set
print(f'\nSource_id containment check: '
      f'{"✓ all downstream IDs came from upstream" if not stray_ids else f"✗ {len(stray_ids)} stray IDs"}')

print()
print('═' * 60)
print('  STAGE SUMMARY')
print('═' * 60)
print(f'  Stage 1 (offline injection):   ✓ complete')
print(f'  Stage 2 (upstream query):      {len(df_upstream)} rows from Gaia')
print(f'  Stage 3 (downstream filtered): {len(df_downstream)} rows, '
      f'filtered from {len(source_ids)} upstream IDs')
print('═' * 60)

Patched build_adql and supervised_pipeline with source_id_filter support.

────────────────────────────────────────────────────────────
STAGE 1 — build_adql injection (offline, no LLM/TAP)
────────────────────────────────────────────────────────────

Without filter:
  SELECT TOP 100 source_id, phot_g_mean_mag, teff_gspphot, ra, dec, bp_rp, pmra, pmdec FROM gaiadr3.gaia_source WHERE teff_gspphot IS NOT NULL AND teff_gspphot BETWEEN 2400 AND 3900 ORDER BY random_index
  ↳ Injected source_id filter: 3 IDs

With 3 IDs:
  SELECT TOP 100 source_id, phot_g_mean_mag, teff_gspphot, ra, dec, bp_rp, pmra, pmdec FROM gaiadr3.gaia_source WHERE source_id IN (123456789, 987654321, 111222333) AND teff_gspphot IS NOT NULL AND teff_gspphot BETWEEN 2400 AND 3900 ORDER BY random_index
  ↳ Injected source_id filter: 200 IDs (capped from 250)

With 250 IDs (cap=200): 200 IDs in ADQL  ✓

source_id IN appears immediately after WHERE: ✓

────────────────────────────────────────────────────────────
STAGE 2 — Up

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.83s/it, est. speed input: 819.47 toks/s, output: 38.84 toks/s]


  Parsed JSON:
{
  "intent": "variability_search",
  "ra": 56.75,
  "dec": 24.12,
  "radius": 1.0,
  "columns": [
    "source_id",
    "ra",
    "dec",
    "phot_g_mean_mag",
    "phot_variable_flag"
  ],
  "filters": {
    "phot_variable_flag": "VARIABLE"
  },
  "join_table": null,
  "limit": 1000
}
  JSON valid.
  ADQL: SELECT TOP 1000 source_id, ra, dec, phot_g_mean_mag, phot_variable_flag, bp_rp, pmra, pmdec FROM gaiadr3.gaia_source WHERE 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 56.75, 24.12, 1.0)) AND phot_variable_flag = 'VARIABLE' ORDER BY random_index
  ADQL syntax valid.
  Evaluating cost …


Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.43s/it, est. speed input: 244.80 toks/s, output: 44.10 toks/s]


  ────────────────────────────────────────────────────
  🟢  COST: CHEAP  (score 20/100)
  ────────────────────────────────────────────────────
    ⚠️  TOP ≤ 1000
    ⚠️  Tight cone ≤ 1.0°
    ⚠️  Indexed filter (CONTAINS)
    ⚠️  No JOIN
    ⚠️  ORDER BY on random_index (not indexed, but not a cost driver)
    💡 No further optimisations needed
  ────────────────────────────────────────────────────
In preparation for Gaia DR4, the Gaia archive is in evolution. Unfortunately, it may be unstable at times and particular types of queries may time out. Please consider registering for a user account (https://www.cosmos.esa.int/web/gaia-users/register). For questions or advice, please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk).

Query sent:
SELECT TOP 1000 source_id, ra, dec, phot_g_mean_mag, phot_variable_flag, bp_rp, pmra, pmdec FROM gaiadr3.gaia_source WHERE 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 56.75, 24.12, 1.0)) AND phot_variable_flag = 'VAR

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.44s/it, est. speed input: 1042.17 toks/s, output: 36.20 toks/s]


  Parsed JSON:
{
  "intent": "hr_diagram",
  "ra": null,
  "dec": null,
  "radius": null,
  "columns": [
    "source_id",
    "parallax",
    "phot_g_mean_mag",
    "bp_rp"
  ],
  "filters": {},
  "join_table": null,
  "limit": 2000
}
  JSON valid.
  ↳ Injected source_id filter: 200 IDs (capped from 234)
  ADQL: SELECT TOP 2000 source_id, parallax, phot_g_mean_mag, bp_rp, ra, dec, pmra, pmdec, (phot_g_mean_mag + 5*LOG10(parallax/100)) AS abs_g_mag FROM gaiadr3.gaia_source WHERE source_id IN (66556535504957568, 66452494217268864, 66509084706321792, 66821105488924416, 65203341630358912, 65289279632597760, 65292986187138432, 65015668739698688, 69807413427563648, 65211038211447552, 66488365784205568, 64948534107307008, 65261929280862208, 65253579864447104, 66792651328276736, 65290688381938688, 65248494623247360, 68322243801359616, 66744174034981760, 64961002395484160, 66816295125702144, 64947984351499776, 65020650901785216, 66761594422163840, 65267078945207424, 65002337161165312, 652601726

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.89s/it, est. speed input: 1514.66 toks/s, output: 31.79 toks/s]


  ────────────────────────────────────────────────────
  🟢  COST: CHEAP  (score 25/100)
  ────────────────────────────────────────────────────
    ⚠️  TOP ≤ 2000
    ⚠️  No spatial filter
    ⚠️  Indexed filters (source_id, parallax, bp_rp)
    ⚠️  ORDER BY random_index
    💡 No further optimisations needed
  ────────────────────────────────────────────────────

Query sent:
SELECT TOP 2000 source_id, parallax, phot_g_mean_mag, bp_rp, ra, dec, pmra, pmdec, (phot_g_mean_mag + 5*LOG10(parallax/100)) AS abs_g_mag FROM gaiadr3.gaia_source WHERE source_id IN (66556535504957568, 66452494217268864, 66509084706321792, 66821105488924416, 65203341630358912, 65289279632597760, 65292986187138432, 65015668739698688, 69807413427563648, 65211038211447552, 66488365784205568, 64948534107307008, 65261929280862208, 65253579864447104, 66792651328276736, 65290688381938688, 65248494623247360, 68322243801359616, 66744174034981760, 64961002395484160, 66816295125702144, 64947984351499776, 65020650901785216, 667

## 10 · Run a query

In [35]:
# Spatial
USER_QUERY = 'Show me stars around the Pleiades'
# USER_QUERY = 'Stars near the galactic centre within 0.5 degrees'

# Photometry
#USER_QUERY = 'Colour histogram of stars near Andromeda'
#USER_QUERY = 'Show me the red dwarfs'
#USER_QUERY = 'Show me the HR diagram and the colour histogram of the Pleiades'

# HR diagram
# USER_QUERY = 'HR diagram of stars near Omega Centauri'

# Stellar population
#USER_QUERY = 'Show me metal-poor giant stars near the galactic centre'
#USER_QUERY = 'Show me metal-poor giant stars'

# Variability
# USER_QUERY = 'Variable stars near the LMC'

# Kinematics
#USER_QUERY = 'How fast do stars near Orion move?'
#USER_QUERY   = 'Show me the 50 nearest stars to the Sun'
#USER_QUERY = 'Pick the fastest stars' 


# Crossmatch
# USER_QUERY = 'Cross-match stars near the galactic centre with astrophysical parameters'


#USER_QUERY = 'Provide me the brightest stars'

def routed_pipeline(user_query: str):
    """
    Top-level entry point.

    1. Router classifies the query as simple or complex.
    2. Simple  → existing supervised_pipeline.
    3. Complex → complex_pipeline (stub for now).

    The classification banner is printed BEFORE the underlying pipeline's
    own banner, so the log reads top-down: route → execute.
    """
    print(f"\n{'═'*60}")
    print(f'  USER QUERY: {user_query}')
    print(f"{'═'*60}")

    print('\n[Router] Classifying query …')
    routing = route_query(user_query)
    verdict = routing['complexity']
    reason  = routing['reason']

    icon = '🟢' if verdict == 'simple' else '🟣'
    print(f'  {icon}  Verdict: {verdict.upper()}')
    print(f'      Reason: {reason}\n')

    if verdict == 'simple':
        return supervised_pipeline(user_query)
    else:
        return complex_pipeline(user_query)


try:
    result  = routed_pipeline(USER_QUERY)
    out     = result['output_json']
    results = result['results']

    # Print summary
    s = out['summary']
    print(f"\n{'═'*60}")
    print(f"  RESULT  [{out['routed_as'].upper()} / {out['composition'].upper()}]")
    print(f"  Steps : {s['total_steps']}  ✓ {s['successful']}  ✗ {s['failed']}")
    print(f"  Time  : {s['total_duration_seconds']}s")
    print(f"{'═'*60}\n")

    # Display each DataFrame
    for key, df in results.items():
        if df is not None:
            print(f'--- Result {key} ({len(df)} rows) ---')
            display(df.head(10))

except RuntimeError as e:
    print(f'\n🔴  {e}')


════════════════════════════════════════════════════════════
  USER QUERY: Show me stars around the Pleiades
════════════════════════════════════════════════════════════

[Router] Classifying query …


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.07it/s, est. speed input: 1071.11 toks/s, output: 33.15 toks/s]


  🟢  Verdict: SIMPLE
      Reason: single cone search


  USER QUERY: Show me stars around the Pleiades

[Attempt 1/3] Parsing …


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.59s/it, est. speed input: 942.85 toks/s, output: 37.11 toks/s]


  Parsed JSON:
{
  "intent": "cone_search",
  "ra": 56.75,
  "dec": 24.12,
  "radius": 1.0,
  "columns": [
    "source_id",
    "ra",
    "dec",
    "phot_g_mean_mag"
  ],
  "filters": {},
  "join_table": null,
  "limit": 1000
}
  JSON valid.
  ADQL: SELECT TOP 1000 source_id, ra, dec, phot_g_mean_mag, bp_rp, pmra, pmdec FROM gaiadr3.gaia_source WHERE 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 56.75, 24.12, 1.0)) ORDER BY random_index
  ADQL syntax valid.
  Evaluating cost …


Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.47s/it, est. speed input: 235.35 toks/s, output: 44.15 toks/s]


  ────────────────────────────────────────────────────
  🟢  COST: CHEAP  (score 20/100)
  ────────────────────────────────────────────────────
    ⚠️  TOP ≤ 1000
    ⚠️  Tight cone ≤ 1.0°
    ⚠️  Indexed filter (CONTAINS)
    ⚠️  No JOIN
    ⚠️  ORDER BY random_index (not indexed, but not a cost driver in this case)
    💡 No further optimisations needed
  ────────────────────────────────────────────────────

Query sent:
SELECT TOP 1000 source_id, ra, dec, phot_g_mean_mag, bp_rp, pmra, pmdec FROM gaiadr3.gaia_source WHERE 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 56.75, 24.12, 1.0)) ORDER BY random_index

INFO: Query finished. [astroquery.utils.tap.core]
Got 1,000 rows

  ✓  1000 rows in 5.89s
  Output JSON → ./pipeline_outputs/show_me_stars_around_the_pleiades.json

════════════════════════════════════════════════════════════
  RESULT  [SIMPLE / SIMPLE]
  Steps : 1  ✓ 1  ✗ 0
  Time  : 5.89s
════════════════════════════════════════════════════════════

--- Result 1 (1000 rows) -

,source_id,ra,dec,phot_g_mean_mag,bp_rp,pmra,pmdec
0,65170528079945984,56.401759,23.518676,19.054289,2.064264,3.077967,-2.793100
1,69817201658236288,56.514012,24.927902,18.234938,1.648279,1.368850,-1.605669
2,66506331628024832,57.300875,23.886588,8.101668,0.296336,19.620729,-45.173239
3,68269471539578240,55.879030,24.302657,15.825923,1.156813,5.913426,-1.465584
4,65266700987253504,55.927350,24.262811,19.298132,2.836407,17.480791,-12.895002
5,66835192980249600,56.846262,24.897098,20.599745,0.625975,1.194813,-1.848215
6,66529425671692160,57.251529,24.093894,20.367111,1.491135,9.185393,-21.455008
7,66732727945184384,56.635586,24.278700,18.735273,1.180223,0.318564,-0.439097
8,66831099877023232,57.177749,25.013688,18.444952,0.913834,1.286902,-1.040130
9,66794476691711616,56.663395,24.632537,18.547205,0.968925,-0.171548,-1.143023


## 11 · Report

In [36]:
import importlib
import display_html
importlib.reload(display_html)
from display_html import generate_report

generate_report(df, USER_QUERY, output_path='gaia_report.html')

Building report ...
Report saved: gaia_report.html  (1205 KB)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  COMPLEX PIPELINE — TEST SUITE
#
#  Three sections, run in order:
#
#   SECTION 1  — Plan validator unit tests (offline, no LLM, no Gaia)
#                Runs in seconds. Fix all failures here before proceeding.
#
#   SECTION 2  — Decomposer tests (LLM only, no Gaia)
#                Tests the decomposer's plan quality in isolation.
#                Runs ~30s (one LLM call per query, no TAP).
#
#   SECTION 3  — End-to-end integration tests (LLM + Gaia)
#                Runs the full complex_pipeline on real queries.
#                Runs several minutes. Uncomment one test at a time.
#
#  Each section prints a pass/fail summary and stops early if critical
#  failures are found, so you don't waste time running Gaia queries when
#  the validator is already broken.
# ═══════════════════════════════════════════════════════════════════════════

import traceback

# ── Shared test helpers ──────────────────────────────────────────────────────

_PASS = '✓'
_FAIL = '✗'
_WARN = '⚠'

class TestResults:
    def __init__(self, section: str):
        self.section  = section
        self.passed   = 0
        self.failed   = 0
        self.failures = []

    def check(self, name: str, condition: bool, detail: str = ''):
        if condition:
            self.passed += 1
            print(f'  {_PASS}  {name}')
        else:
            self.failed += 1
            self.failures.append((name, detail))
            print(f'  {_FAIL}  {name}')
            if detail:
                print(f'       {detail}')

    def summary(self) -> bool:
        total = self.passed + self.failed
        ok    = self.failed == 0
        icon  = _PASS if ok else _FAIL
        print(f'\n  {icon}  {self.section}: {self.passed}/{total} passed')
        if self.failures:
            for name, detail in self.failures:
                print(f'     • {name}')
                if detail:
                    print(f'       {detail}')
        return ok


# ═══════════════════════════════════════════════════════════════════════════
#  SECTION 1 — Plan validator unit tests (offline)
# ═══════════════════════════════════════════════════════════════════════════
print('═' * 70)
print('  SECTION 1 — Plan validator unit tests (offline)')
print('═' * 70)

r1 = TestResults('Validator')

# ── Helper: a known-good parallel plan ──────────────────────────────────────
def _good_parallel_plan():
    return {
        'shared_region': {'target': 'Pleiades', 'ra': 56.75, 'dec': 24.12, 'radius': 1.0},
        'composition':   'parallel',
        'steps': [
            {
                'step_id': 1, 'intent': 'hr_diagram',
                'description': 'HR diagram', 'natural_language_query': 'HR diagram near the Pleiades',
                'depends_on': [], 'dependency_type': None, 'override_region': None,
                'filters': {}, 'columns': ['source_id', 'parallax', 'phot_g_mean_mag', 'bp_rp'],
                'join_table': None, 'limit': 2000,
            },
            {
                'step_id': 2, 'intent': 'color_histogram',
                'description': 'Colour histogram', 'natural_language_query': 'Colour histogram near the Pleiades',
                'depends_on': [], 'dependency_type': None, 'override_region': None,
                'filters': {}, 'columns': ['source_id', 'bp_rp', 'phot_g_mean_mag'],
                'join_table': None, 'limit': 2000,
            },
        ],
    }

# ── Helper: a known-good sequential plan ────────────────────────────────────
def _good_sequential_plan():
    return {
        'shared_region': {'target': 'Orion', 'ra': 83.82, 'dec': -5.39, 'radius': 2.0},
        'composition':   'sequential',
        'steps': [
            {
                'step_id': 1, 'intent': 'variability_search',
                'description': 'Variable stars near Orion',
                'natural_language_query': 'Variable stars near Orion',
                'depends_on': [], 'dependency_type': None, 'override_region': None,
                'filters': {'phot_variable_flag': 'VARIABLE'},
                'columns': ['source_id', 'ra', 'dec', 'phot_g_mean_mag', 'phot_variable_flag'],
                'join_table': None, 'limit': 200,
            },
            {
                'step_id': 2, 'intent': 'hr_diagram',
                'description': 'HR diagram of variable stars',
                'natural_language_query': 'HR diagram of variable stars near Orion',
                'depends_on': [1], 'dependency_type': 'source_id_filter',
                'override_region': None,
                'filters': {}, 'columns': ['source_id', 'parallax', 'phot_g_mean_mag', 'bp_rp'],
                'join_table': None, 'limit': 1000,
            },
        ],
    }

# ── 1.1  Valid plans pass cleanly ────────────────────────────────────────────
errs = validate_plan(_good_parallel_plan())
r1.check('Valid parallel plan → no errors', errs == [],
         f'Got errors: {errs}')

errs = validate_plan(_good_sequential_plan())
r1.check('Valid sequential plan → no errors', errs == [],
         f'Got errors: {errs}')

# ── 1.2  Top-level structure errors ──────────────────────────────────────────
errs = validate_plan({'composition': 'parallel', 'steps': []})
r1.check('Empty steps list → error',
         any('steps' in e for e in errs))

errs = validate_plan({'composition': 'sideways', 'steps': [_good_parallel_plan()['steps'][0]]})
r1.check('Invalid composition → error',
         any('composition' in e for e in errs))

errs = validate_plan({'composition': 'parallel',
                       'shared_region': {'ra': None, 'dec': 24.0, 'radius': 1.0},
                       'steps': [_good_parallel_plan()['steps'][0]]})
r1.check('shared_region with null ra → error',
         any('shared_region.ra' in e for e in errs))

# ── 1.3  Per-step intent validation ──────────────────────────────────────────
bad = _good_parallel_plan()
bad['steps'][0]['intent'] = 'time_travel'
errs = validate_plan(bad)
r1.check('Unknown intent → error',
         any('intent' in e and 'time_travel' in e for e in errs))

# ── 1.4  Cone-required intents need a region ─────────────────────────────────
bad = _good_parallel_plan()
bad['shared_region'] = None   # remove shared region
bad['steps'][1]['override_region'] = None   # color_histogram has no region
errs = validate_plan(bad)
r1.check('color_histogram without any region → error',
         any('spatial region' in e for e in errs))

# Cone-required but with override_region — should be fine
ok_plan = _good_parallel_plan()
ok_plan['shared_region'] = None
ok_plan['steps'][1]['override_region'] = {'ra': 56.75, 'dec': 24.12, 'radius': 1.0}
# step[0] is hr_diagram (not cone-required) so also needs no region — set sky-wide
ok_plan['steps'][0]['natural_language_query'] = 'HR diagram of nearby stars'
errs = validate_plan(ok_plan)
r1.check('cone_required with override_region and no shared_region → no error',
         not any('spatial region' in e for e in errs),
         f'Unexpected errors: {errs}')

# ── 1.5  depends_on / dependency_type consistency ────────────────────────────
bad = _good_sequential_plan()
bad['steps'][1]['dependency_type'] = None   # missing when depends_on is set
errs = validate_plan(bad)
r1.check('depends_on set but dependency_type null → error',
         any('dependency_type' in e for e in errs))

bad = _good_parallel_plan()
bad['steps'][0]['dependency_type'] = 'source_id_filter'  # set when depends_on empty
errs = validate_plan(bad)
r1.check('depends_on empty but dependency_type set → error',
         any('dependency_type' in e for e in errs))

# ── 1.6  composition vs depends_on consistency ───────────────────────────────
bad = _good_parallel_plan()
bad['steps'][1]['depends_on'] = [1]   # parallel plan but step has dependency
bad['steps'][1]['dependency_type'] = 'source_id_filter'
errs = validate_plan(bad)
r1.check("composition='parallel' but step has depends_on → error",
         any('parallel' in e for e in errs))

bad = _good_sequential_plan()
bad['steps'][1]['depends_on'] = []    # sequential plan but no step has dependency
bad['steps'][1]['dependency_type'] = None
errs = validate_plan(bad)
r1.check("composition='sequential' but no step has depends_on → error",
         any('sequential' in e for e in errs))

# ── 1.7  DAG checks ──────────────────────────────────────────────────────────
# Reference to nonexistent step_id
bad = _good_sequential_plan()
bad['steps'][1]['depends_on'] = [99]
errs = validate_plan(bad)
r1.check('depends_on references nonexistent step_id → error',
         any('unknown step_id' in e for e in errs))

# Cycle: step 1 → step 2 → step 1
cyclic = {
    'shared_region': {'target': 'Orion', 'ra': 83.82, 'dec': -5.39, 'radius': 1.0},
    'composition': 'sequential',
    'steps': [
        {
            'step_id': 1, 'intent': 'variability_search',
            'description': 'A', 'natural_language_query': 'Variable stars near Orion',
            'depends_on': [2], 'dependency_type': 'source_id_filter',
            'override_region': None,
            'filters': {}, 'columns': ['source_id', 'phot_g_mean_mag'],
            'join_table': None, 'limit': 200,
        },
        {
            'step_id': 2, 'intent': 'hr_diagram',
            'description': 'B', 'natural_language_query': 'HR diagram near Orion',
            'depends_on': [1], 'dependency_type': 'source_id_filter',
            'override_region': None,
            'filters': {}, 'columns': ['source_id', 'parallax', 'phot_g_mean_mag', 'bp_rp'],
            'join_table': None, 'limit': 1000,
        },
    ],
}
errs = validate_plan(cyclic)
r1.check('Cycle in dependencies → error',
         any('Cycle' in e for e in errs))

# Duplicate step_id
bad = _good_parallel_plan()
bad['steps'][1]['step_id'] = 1   # duplicate
errs = validate_plan(bad)
r1.check('Duplicate step_id → error',
         any('Duplicate' in e for e in errs))

# ── 1.8  Upstream must expose source_id ─────────────────────────────────────
bad = _good_sequential_plan()
bad['steps'][0]['columns'] = ['ra', 'dec', 'phot_g_mean_mag']   # no source_id
errs = validate_plan(bad)
r1.check('Upstream step missing source_id → error',
         any('source_id' in e for e in errs))

# ── 1.9  join_table rules ────────────────────────────────────────────────────
crossmatch_plan = {
    'shared_region': {'target': 'LMC', 'ra': 80.9, 'dec': -69.8, 'radius': 2.0},
    'composition': 'parallel',
    'steps': [{
        'step_id': 1, 'intent': 'internal_crossmatch',
        'description': 'Crossmatch with Cepheids',
        'natural_language_query': 'Cross-match stars near the LMC with the Cepheid table',
        'depends_on': [], 'dependency_type': None, 'override_region': None,
        'filters': {}, 'columns': ['source_id', 'ra', 'dec'],
        'join_table': 'vari_cepheid', 'limit': 1000,
    }],
}
errs = validate_plan(crossmatch_plan)
r1.check('Valid internal_crossmatch → no errors', errs == [],
         f'Got: {errs}')

bad = crossmatch_plan.copy()
bad['steps'] = [dict(bad['steps'][0], join_table=None)]
errs = validate_plan(bad)
r1.check('internal_crossmatch without join_table → error',
         any('join_table' in e for e in errs))

bad['steps'] = [dict(bad['steps'][0], join_table='made_up_table')]
errs = validate_plan(bad)
r1.check('internal_crossmatch with invalid join_table → error',
         any('valid DR3' in e or 'not a valid' in e for e in errs))

# ── Section 1 summary ────────────────────────────────────────────────────────
s1_ok = r1.summary()
if not s1_ok:
    print('\n  ⛔  Fix validator failures before running Sections 2 and 3.')


# ═══════════════════════════════════════════════════════════════════════════
#  SECTION 2 — Decomposer tests (LLM only, no Gaia)
#  Only runs if Section 1 passed.
# ═══════════════════════════════════════════════════════════════════════════
if s1_ok:
    print('\n' + '═' * 70)
    print('  SECTION 2 — Decomposer tests (LLM only, no Gaia)')
    print('═' * 70)

    r2 = TestResults('Decomposer')

    # For each case: call decompose_query, validate the plan, then check
    # specific properties we care about — composition type, number of steps,
    # intents used, whether shared_region is set, whether dependencies are set.

    DECOMPOSER_CASES = [
        {
            'label':             'Parallel same-region: HR + histogram of Pleiades',
            'query':             'Show me the HR diagram and the colour histogram of the Pleiades',
            'expect_composition': 'parallel',
            'expect_n_steps':     2,
            'expect_intents':     {'hr_diagram', 'color_histogram'},
            'expect_shared_region': True,
            'expect_any_deps':    False,
        },
        {
            'label':             'Parallel different-regions: compare Andromeda vs LMC',
            'query':             'Compare colour distributions near Andromeda and the LMC',
            'expect_composition': 'parallel',
            'expect_n_steps':     2,
            'expect_intents':     {'color_histogram'},
            'expect_shared_region': False,   # two different regions → no shared
            'expect_any_deps':    False,
        },
        {
            'label':             'Parallel sky-wide: red dwarfs and white dwarfs',
            'query':             'Show me red dwarfs and also white dwarfs',
            'expect_composition': 'parallel',
            'expect_n_steps':     2,
            'expect_intents':     {'stellar_population'},
            'expect_shared_region': False,
            'expect_any_deps':    False,
        },
        {
            'label':             'Sequential 2-step: variable stars → HR diagram',
            'query':             'Find variable stars near Orion and plot their HR diagram',
            'expect_composition': 'sequential',
            'expect_n_steps':     2,
            'expect_intents':     {'variability_search', 'hr_diagram'},
            'expect_shared_region': True,
            'expect_any_deps':    True,
        },
        {
            'label':             'Sequential 3-step: variable → crossmatch → HR diagram',
            'query':             'Find variable stars near Orion, cross-match with the Cepheid table, and plot their HR diagram',
            'expect_composition': 'sequential',
            'expect_n_steps':     3,
            'expect_intents':     {'variability_search', 'internal_crossmatch', 'hr_diagram'},
            'expect_shared_region': True,
            'expect_any_deps':    True,
        },
    ]

    for case in DECOMPOSER_CASES:
        print(f"\n  ── {case['label']}")
        try:
            plan   = decompose_query(case['query'])
            errors = validate_plan(plan)

            # Must pass validation
            r2.check(f'  Plan valid (no validator errors)',
                     errors == [], f'Errors: {errors}')

            # Composition
            r2.check(f'  composition = {case["expect_composition"]!r}',
                     plan.get('composition') == case['expect_composition'],
                     f'Got: {plan.get("composition")!r}')

            # Step count
            n = len(plan.get('steps', []))
            r2.check(f'  {n} steps (expected {case["expect_n_steps"]})',
                     n == case['expect_n_steps'],
                     f'Got {n} steps')

            # Intents used
            got_intents = {s['intent'] for s in plan.get('steps', [])}
            r2.check(f'  Expected intents present: {case["expect_intents"]}',
                     case['expect_intents'].issubset(got_intents),
                     f'Got intents: {got_intents}')

            # shared_region
            has_shared = plan.get('shared_region') is not None
            if case['expect_shared_region']:
                r2.check('  shared_region is set', has_shared,
                         f'shared_region = {plan.get("shared_region")}')
            else:
                r2.check('  shared_region is null (different regions)', not has_shared,
                         f'shared_region = {plan.get("shared_region")}')

            # dependencies
            any_deps = any(len(s.get('depends_on', [])) > 0 for s in plan.get('steps', []))
            if case['expect_any_deps']:
                r2.check('  At least one step has depends_on set', any_deps)
            else:
                r2.check('  No step has depends_on (pure parallel)', not any_deps,
                         f'Steps with deps: {[s["step_id"] for s in plan["steps"] if s.get("depends_on")]}')

            # For sequential: upstream steps with dependants expose source_id
            if case['expect_any_deps']:
                steps_with_dependants = {
                    dep_id
                    for s in plan['steps']
                    for dep_id in s.get('depends_on', [])
                }
                step_map = {s['step_id']: s for s in plan['steps']}
                upstream_ok = all(
                    'source_id' in step_map[sid].get('columns', [])
                    for sid in steps_with_dependants
                    if sid in step_map
                )
                r2.check('  Upstream steps include source_id in columns', upstream_ok,
                         f'Steps missing source_id: '
                         f'{[sid for sid in steps_with_dependants if "source_id" not in step_map.get(sid, {}).get("columns", [])]}')

            # Print the plan for manual inspection
            print(f'\n    Plan summary:')
            for s in plan['steps']:
                dep_str = f' → depends on {s["depends_on"]}' if s.get('depends_on') else ''
                print(f'      Step {s["step_id"]}: {s["intent"]:25} | {s["description"][:40]}{dep_str}')

        except Exception as exc:
            r2.check(f'  decompose_query() raised exception', False,
                     traceback.format_exc())

    s2_ok = r2.summary()
    if not s2_ok:
        print('\n  ⚠️  Decomposer failures — inspect plan output above and tune the prompt.')
else:
    s2_ok = False
    print('\n  ⊘  Section 2 skipped (Section 1 failed)')


# ═══════════════════════════════════════════════════════════════════════════
#  SECTION 3 — End-to-end integration tests (LLM + Gaia)
#
#  These hit the Gaia TAP server and take several minutes total.
#  Uncomment ONE test at a time to avoid hammering the server.
#  Read the results carefully — row counts and query shapes matter.
# ═══════════════════════════════════════════════════════════════════════════
if s2_ok:
    print('\n' + '═' * 70)
    print('  SECTION 3 — End-to-end integration tests (LLM + Gaia)')
    print('═' * 70)

    r3 = TestResults('End-to-end')

    def _run_e2e(label: str, query: str,
                 expect_n_steps: int,
                 expect_min_rows_per_step: int = 1) -> bool:
        """Run complex_pipeline and do basic sanity checks on the results."""
        print(f'\n  ── {label}')
        print(f'     Query: {query}')
        try:
            out     = complex_pipeline(query, save_json=True)
            results = out['results']
            summary = out['output_json']['summary']

            r3.check(f'  Pipeline did not raise',        True)
            r3.check(f'  {summary["successful"]}/{expect_n_steps} steps succeeded',
                     summary['successful'] == expect_n_steps,
                     f'successful={summary["successful"]}, failed={summary["failed"]}, '
                     f'skipped={summary["skipped"]}')

            # Per-step checks
            for sid, df in results.items():
                if df is None:
                    r3.check(f'  Step {sid} returned a DataFrame', False,
                             'Got None')
                    continue
                r3.check(f'  Step {sid}: {len(df)} rows (≥{expect_min_rows_per_step})',
                         len(df) >= expect_min_rows_per_step,
                         f'Got {len(df)} rows')
                r3.check(f'  Step {sid}: has source_id column',
                         'source_id' in df.columns,
                         f'Columns: {list(df.columns)}')

            # For sequential: downstream results should be a subset of upstream
            plan  = out['output_json']['plan']
            steps = plan['steps']
            for step in steps:
                if step.get('depends_on'):
                    for dep_id in step['depends_on']:
                        upstream_df   = results.get(dep_id)
                        downstream_df = results.get(step['step_id'])
                        if upstream_df is not None and downstream_df is not None:
                            up_ids   = set(upstream_df['source_id'].dropna().astype(int))
                            down_ids = set(downstream_df['source_id'].dropna().astype(int))
                            stray    = down_ids - up_ids
                            r3.check(
                                f'  Step {step["step_id"]} IDs ⊆ Step {dep_id} IDs '
                                f'(containment check)',
                                len(stray) == 0,
                                f'{len(stray)} stray IDs found'
                            )

            return True

        except Exception as exc:
            r3.check(f'  Pipeline raised exception', False,
                     traceback.format_exc())
            return False


    # ── Test 3.1: Parallel same-region ──────────────────────────────────────
    # Uncomment to run:
    _run_e2e(
        label          = 'Parallel: HR diagram + colour histogram of Pleiades',
        query          = 'Show me the HR diagram and the colour histogram of the Pleiades',
        expect_n_steps = 2,
        expect_min_rows_per_step = 10,
    )

    # ── Test 3.2: Parallel different-regions ────────────────────────────────
    # Uncomment to run (after 3.1 passes):
    # _run_e2e(
    #     label          = 'Parallel: colour histogram Andromeda vs LMC',
    #     query          = 'Compare colour distributions near Andromeda and the LMC',
    #     expect_n_steps = 2,
    #     expect_min_rows_per_step = 10,
    # )

    # ── Test 3.3: Sequential 2-step ─────────────────────────────────────────
    # Uncomment to run (after 3.2 passes):
    # _run_e2e(
    #     label          = 'Sequential: variable stars near Orion → HR diagram',
    #     query          = 'Find variable stars near Orion and plot their HR diagram',
    #     expect_n_steps = 2,
    #     expect_min_rows_per_step = 1,
    # )

    # ── Test 3.4: Sequential 3-step (hardest) ───────────────────────────────
    # Uncomment to run (after 3.3 passes):
    # _run_e2e(
    #     label          = 'Sequential 3-step: variable → Cepheid crossmatch → HR diagram',
    #     query          = 'Find variable stars near Orion, cross-match with the Cepheid table, and plot their HR diagram',
    #     expect_n_steps = 3,
    #     expect_min_rows_per_step = 1,
    # )

    r3.summary()
else:
    print('\n  ⊘  Section 3 skipped (Sections 1 or 2 failed)')


# ═══════════════════════════════════════════════════════════════════════════
#  FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════
print('\n' + '═' * 70)
print('  OVERALL TEST SUMMARY')
print('═' * 70)
print(f'  Section 1 (Validator):    {"PASS" if s1_ok else "FAIL"}')
print(f'  Section 2 (Decomposer):   {"PASS" if s2_ok else ("FAIL" if s1_ok else "SKIPPED")}')
print(f'  Section 3 (End-to-end):   {"see above" if s2_ok else "SKIPPED"}')

<h1> TEST - MARKDOWN 

In [76]:
# ── Test queries with expected classifications ───────────────────────────────
TEST_QUERIES = [
    # ── Should be SIMPLE ─────────────────────────────────────────────────────
    ('simple', 'Show me the brightest stars'),
    ('simple', 'HR diagram of Omega Centauri'),
    ('simple', "Red dwarfs near Barnard's star"),
    ('simple', 'Cross-match stars near the LMC with astrophysical parameters'),
    ('simple', 'Nearest 50 stars to the Sun'),
    ('simple', 'Stars near Orion that are also variable'),
    ('simple', 'Show me metal-poor giant stars near the galactic centre'),
    ('simple', 'Colour histogram of stars near Andromeda'),
    ('simple', 'How fast do stars near Orion move?'),
    ('simple', 'Variable stars near the LMC'),
 
    # ── Should be COMPLEX ────────────────────────────────────────────────────
    ('complex', 'Show me the HR diagram and the colour histogram of the Pleiades'),
    ('complex', 'Compare colour distributions near Andromeda and the LMC'),
    ('complex', 'Find variable stars near Orion and compute their kinematics'),
    ('complex', 'Show me red dwarfs and also white dwarfs'),
    ('complex', 'Show me the nearest 50 stars and their HR diagram'),
    ('complex', 'Cross-match variable stars near M31 with the Cepheid table and plot their HR diagram'),
    ('complex', 'Show me Cepheids and RR Lyrae stars'),
    ('complex', 'HR diagram of the Pleiades and HR diagram of Omega Centauri'),
]
 
 
def run_router_tests():
    """Run all test queries through the router and report accuracy."""
    print('═' * 80)
    print('  ROUTER TEST SUITE')
    print('═' * 80)
    print(f'  {len(TEST_QUERIES)} queries to classify\n')
 
    results = []
    for expected, query in TEST_QUERIES:
        routing = route_query(query)
        actual  = routing['complexity']
        reason  = routing['reason']
        match   = (actual == expected)
        results.append((expected, actual, query, reason, match))
 
    # ── Print table ──────────────────────────────────────────────────────────
    print(f'{"✓/✗":<4} {"EXPECTED":<10} {"ACTUAL":<10} QUERY')
    print('─' * 80)
    for expected, actual, query, reason, match in results:
        mark = '✓' if match else '✗'
        # truncate long queries for table display
        q_short = query if len(query) <= 50 else query[:47] + '...'
        print(f'{mark:<4} {expected:<10} {actual:<10} {q_short}')
        if not match:
            print(f'     └─ reason: {reason}')
 
    # ── Summary ──────────────────────────────────────────────────────────────
    total      = len(results)
    correct    = sum(1 for r in results if r[4])
    accuracy   = 100 * correct / total
    simple_total   = sum(1 for r in results if r[0] == 'simple')
    simple_correct = sum(1 for r in results if r[0] == 'simple' and r[4])
    complex_total   = sum(1 for r in results if r[0] == 'complex')
    complex_correct = sum(1 for r in results if r[0] == 'complex' and r[4])
 
    print()
    print('─' * 80)
    print(f'  OVERALL:  {correct}/{total}  ({accuracy:.0f}%)')
    print(f'  Simple:   {simple_correct}/{simple_total}  '
          f'({100*simple_correct/simple_total:.0f}%)')
    print(f'  Complex:  {complex_correct}/{complex_total}  '
          f'({100*complex_correct/complex_total:.0f}%)')
    print('─' * 80)
 
    # Failure breakdown by direction
    false_complex = [r for r in results if r[0] == 'simple'  and r[1] == 'complex']
    false_simple  = [r for r in results if r[0] == 'complex' and r[1] == 'simple']
    if false_complex:
        print(f'\n  ⚠️  {len(false_complex)} false-COMPLEX (over-routing):')
        for _, _, q, reason, _ in false_complex:
            print(f'      • "{q}"')
            print(f'        router said: {reason}')
    if false_simple:
        print(f'\n  ⚠️  {len(false_simple)} false-SIMPLE (under-routing):')
        for _, _, q, reason, _ in false_simple:
            print(f'      • "{q}"')
            print(f'        router said: {reason}')
 
    return results
 
 
# ── Run it ───────────────────────────────────────────────────────────────────
test_results = run_router_tests()

════════════════════════════════════════════════════════════════════════════════
  ROUTER TEST SUITE
════════════════════════════════════════════════════════════════════════════════
  18 queries to classify



Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.63it/s, est. speed input: 852.97 toks/s, output: 37.44 toks/s]

✓/✗  EXPECTED   ACTUAL     QUERY
────────────────────────────────────────────────────────────────────────────────
✓    simple     simple     Show me the brightest stars
✓    simple     simple     HR diagram of Omega Centauri
✓    simple     simple     Red dwarfs near Barnard's star
✓    simple     simple     Cross-match stars near the LMC with astrophysic...
✓    simple     simple     Nearest 50 stars to the Sun
✓    simple     simple     Stars near Orion that are also variable
✓    simple     simple     Show me metal-poor giant stars near the galacti...
✓    simple     simple     Colour histogram of stars near Andromeda
✓    simple     simple     How fast do stars near Orion move?
✓    simple     simple     Variable stars near the LMC
✓    complex    complex    Show me the HR diagram and the colour histogram...
✓    complex    complex    Compare colour distributions near Andromeda and...
✓    complex    complex    Find variable stars near Orion and compute thei...
✓    complex    comp

In [72]:
from ipynb.fs.full.Complex import double

4
